In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/10.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/100.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1000.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1001.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1002.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1003.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1004.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1005.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1006.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1007.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1008.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/1009.mat
/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data/101.mat
/kaggle/input

# 1.Preprocessing & Data Loader (Tiền xử lý & Nạp dữ liệu)

## 1.1 Khai báo Thư viện & Thiết lập Hằng số Cấu hình (Hyperparameters)

### Khai báo các thông số
* KICH_THUOC = 256 Giúp resize về kích thước tiêu chuẩn nhưng vẫn giữ đầy đủ các chi tiết mô, làm cho quá trình tính toán ma trận diễn ra trơn tru tránh lỗi kích thước số lẻ
* SO_FILE = 3064 Tổng số file trong dataset
* TI_LE_TRAIN = 0.7 Đảm bảo cung cấp đủ lượng dữ liệu lớn và đa dạng để mô hình học sâu (với hàng triệu tham số) có thể tổng quát hóa các đặc trưng hình thái của khối u.
* TI_LE_VAL = 0.15 Dùng để kiểm định trong quá trình train, giúp tính toán hàm mất mát (loss) qua mỗi epoch nhằm chọn mô hình tốt nhất (best model) và áp dụng cơ chế dừng sớm (early stopping) chống quá khớp (overfitting).
* 15% còn lại Giữ hoàn toàn "mù" (chưa từng thấy qua trong quá trình huấn luyện và tinh chỉnh) để đánh giá khách quan hiệu suất cuối cùng của mô hình
* HAT_GIONG = 42 Đảm bảo tính lặp lại (reproducibility) của nghiên cứu. Trong lập trình, các hàm xáo trộn dữ liệu (shuffle) hay chia tập train/val/test đều mang tính ngẫu nhiên.
* GOC_XOAY_TOI_DA = 15 Phép biến đổi xoay góc nhỏ giúp mô hình thích nghi tốt với sự dịch chuyển góc độ này.Không thể chọn 90$^\circ$ hoặc 80$^\circ$ vì nó sẽ làm biến dạng không gian giải phẩu thực tế của não bộ.Góc xoay giới hạn ở mức $\pm 15^\circ$ giúp tăng cường dữ liệu (Data Augmentation) hiệu quả mà không làm hỏng ranh giới hoặc bóp méo hình dạng tự nhiên của khối u.

In [2]:
import os                           
import random                      
import cv2                        
import h5py                         
import numpy as np                  
import torch
import matplotlib.pyplot as plt     # matplotlib: hiển thị ảnh
from torch.utils.data import Dataset, DataLoader

THU_MUC_DU_LIEU = '/kaggle/input/datasets/duonghuy2211/mri-pipeline-dataset/data'    # thư mục chứa các file 1.mat ... 3064.mat
SO_FILE = 3064              # tổng số file .mat
KICH_THUOC = 256            # kích thước ảnh sau khi resize (256 x 256)
TI_LE_TRAIN = 0.7           # 70% bệnh nhân cho tập Train
TI_LE_VAL = 0.15            # 15% bệnh nhân cho tập Val, phần còn lại cho Test
HAT_GIONG = 42              # số cố định cho random để lần nào chạy cũng chia giống nhau
GOC_XOAY_TOI_DA = 15        # augmentation xoay ngẫu nhiên từ -15 đến 15 độ

## 1.2 Hàm Đọc Dữ liệu Định dạng `.mat` (`doc_file_mat`)
### Hàm dùng để đọc các dữ liệu gốc từ tập dataset
* **Sử dụng thư viện `h5py`**
* **Hiệu chỉnh hướng ma trận**:MATLAB lưu trữ ma trận theo thứ tự cột trước hàng sau (column-major order), trong khi Python đọc theo hàng trước cột sau (row-major order). Phép chuyển vị `.T` giúp đưa ảnh về đúng không gian giải phẫu ban đầu, tránh bị lật ngược.
* **Chuyển đổi dữ liệu**: Ảnh được ép sang `float32` để phục vụ các bước chuẩn hóa cường độ sau này, trong khi mask được giữ ở định dạng nhị phân

In [3]:
def doc_file_mat(duong_dan):
    # cjdata là tên nhóm dữ liệu trong file .mat (do người tạo bộ dữ liệu đặt),
    # bên trong có image (ảnh MRI) và tumorMask (mask khối u)
    du_lieu = h5py.File(duong_dan, 'r')     # 'r' = chỉ đọc

    anh = np.array(du_lieu['cjdata']['image'])
    anh = anh.T                             # .T sửa hướng ảnh vì MATLAB lưu theo cột, Python đọc theo hàng
    anh = anh.astype(np.float32)            # đổi sang số thực để tính Min-Max chính xác

    mask = np.array(du_lieu['cjdata']['tumorMask'])
    mask = mask.T
    mask = mask.astype(np.uint8)            # mask chỉ có 0 và 1

    du_lieu.close()
    return anh, mask


## 1.3 Hàm Trích xuất Mã Bệnh Nhân (`doc_ma_benh_nhan`)
Hàm này dùng để đọc định danh bệnh nhân (Patient ID - PID) từ cấu trúc file `.mat`:
* **Xử lý định dạng gốc**: Trong bộ dataset Figshare, mã PID không lưu dưới dạng chuỗi văn bản thông thường mà lưu thành một mảng các ký tự/mã số ASCII (ví dụ: `[49 48 48...]`).
* **Làm phẳng và chuyển đổi**: Hàm thực hiện làm phẳng mảng (`flatten()`), duyệt qua từng phần tử và ép kiểu sang số nguyên để chuyển hóa thành chuỗi định danh duy nhất cho từng bệnh nhân.
* **Ý nghĩa ứng dụng**: Việc trích xuất chính xác mã bệnh nhân là cơ sở cực kỳ quan trọng để thực hiện chiến lược **chia tập dữ liệu theo bệnh nhân (Patient-wise splitting)**, đảm bảo các lát cắt của cùng một bệnh nhân không bị phân rải vào cả tập Train và Test, từ đó ngăn chặn hiện tượng rò rỉ dữ liệu (data leakage) giúp mô hình đánh giá khách quan nhất.

In [4]:
def doc_ma_benh_nhan(duong_dan):
    # PID được lưu dạng dãy mã số, ví dụ file 1.mat: [49 48 48 51 54 48]
    # Ta nối các số lại thành chuỗi "49 48 48 51 54 48 " để làm mã bệnh nhân
    du_lieu = h5py.File(duong_dan, 'r')
    day_so = np.array(du_lieu['cjdata']['PID'])
    day_so = day_so.flatten()               # bảng 6x1 -> 1 hàng 6 số
    du_lieu.close()

    pid = '' 
    for so in day_so:
        so_nguyen = int(so)                 # đổi số của numpy (49.0) thành số nguyên (49)
        pid = pid + str(so_nguyen) + ' '
    return pid


## 1.4 Hàm Chuẩn hóa Kích thước Không gian (`resize`)
Hàm này thực hiện đồng bộ hóa kích thước của ảnh MRI và mặt nạ khối u về chuẩn `KICH_THUOC` ($\text{KICH\_THUOC} \times \text{KICH\_THUOC}$):
* **Đối với ảnh MRI (`cv2.INTER_LINEAR` - Nội suy song tuyến tính)**: Giúp làm mượt các giá trị điểm ảnh xung quanh, giữ lại cường độ xám liên tục và tự nhiên của mô não.
* **Đối với mặt nạ nhãn (`cv2.INTER_NEAREST` - Nội suy lân cận gần nhất)**: Bắt buộc phải dùng phương pháp này để **bảo toàn tính nhị phân tuyệt đối (chỉ chứa giá trị 0 hoặc 1)**. Nếu dùng nội suy tuyến tính, các điểm ảnh biên sẽ bị chia dạng thập phân (ví dụ: $0.5$), làm biến dạng ranh giới khối u và gây lỗi trong quá trình tính toán hàm mất mát hoặc chỉ số đánh giá.

In [5]:
def resize(anh, mask):
    # INTER_LINEAR (lấy trung bình xung quanh , làm ảnh mượt hơn)
    anh = cv2.resize(anh, (KICH_THUOC, KICH_THUOC), interpolation=cv2.INTER_LINEAR)
    # Mask: INTER_NEAREST (lấy giá trị của điểm gần nhất, chỉ có 0 và 1.
    #  Ta không dùng linear vì sẽ tạo ra điểm ảnh 0,5 nghĩa là nửa u nửa bth, gây ra lỗi và không đoán được)
    mask = cv2.resize(mask, (KICH_THUOC, KICH_THUOC), interpolation=cv2.INTER_NEAREST)
    return anh, mask


## 1.5 Hàm Chuẩn hóa Cường độ Điểm ảnh (`chuan_hoa`)
Hàm này thực hiện việc chuẩn hóa độ tương phản và khoảng giá trị cường độ sáng của ảnh MRI:
* **Thuật toán Min-Max Scaling**: Đưa dải giá trị cường độ xám của ảnh gốc về đoạn $[0, 255]$ dựa trên giá trị nhỏ nhất (`nho_nhat`) và lớn nhất (`lon_nhat`) của từng lát cắt.
* **Xử lý ngoại lệ an toàn**: Bổ sung điều kiện kiểm tra trường hợp ảnh bị phẳng hoàn toàn (`lon_nhat == nho_nhat`, ví dụ ảnh đen tuyền) để tránh lỗi chia cho số 0 (`ZeroDivisionError`).
* **Làm tròn và ép kiểu**: Dùng hàm `np.round()` kết hợp ép kiểu `uint8` để chuyển đổi các giá trị số thực sau khi tính toán về dạng số nguyên 8-bit chuẩn, giúp tối ưu hóa bộ nhớ và tương thích hoàn hảo với các bước xử lý ảnh phía sau (như thuật toán ngưỡng Otsu).

In [6]:
def chuan_hoa(anh):
    nho_nhat = anh.min()
    lon_nhat = anh.max()

    if lon_nhat == nho_nhat:
        return np.zeros(anh.shape, dtype=np.uint8)

    anh = (anh - nho_nhat) / (lon_nhat - nho_nhat) * 255
    anh = np.round(anh)                  # round là hàm giúp làm tròn số 
    anh = anh.astype(np.uint8)           
    return anh

## 1.6 Các Bộ Lọc Khử Nhiễu & Tăng Cường Tương Phản
Nhóm hàm này thực hiện các phép xử lý tín hiệu số trên ảnh MRI nhằm làm sạch nhiễu và làm nổi bật cấu trúc giải phẫu:
* **`median(anh, k=3)` (Bộ lọc trung vị - Median Blur)**: Sử dụng cửa sổ kích thước $k=3$, rất hiệu quả trong việc loại bỏ nhiễu hạt muối tiêu (salt-and-pepper noise) thường xuất hiện trong ảnh y tế mà vẫn **bảo toàn nguyên vẹn các đường biên sắc nét** của khối u.
* **`gauss(anh, k=3)` (Bộ lọc Gauss - Gaussian Blur)**: Làm mượt ảnh bằng phân phối chuẩn, giúp giảm bớt nhiễu nền tần số cao trước khi đưa vào các bước phát hiện biên.
* **`can_bang_histogram(anh)` (Cân bằng lược đồ màu - Histogram Equalization)**: Mở rộng dải động (dynamic range) của cường độ xám, giúp cải thiện độ tương phản tổng thể, làm cho các vùng mô tổn thương có độ sáng mờ trở nên rõ nét hơn so với mô não lành xung quanh.

In [7]:
def median(anh, k=3):
    return cv2.medianBlur(anh, k)

def gauss(anh, k=3):
    return cv2.GaussianBlur(anh, (k, k), 0)

def can_bang_histogram(anh):
    return cv2.equalizeHist(anh)

## 1.7 Hàm Xây Dựng Quy Trình Tiền Xử Lý Tổng Hợp (`tien_xu_ly`)
Hàm này đóng vai trò là "đường ống" (pipeline) tự động kết hợp toàn bộ các bước chuẩn bị dữ liệu cho một lát cắt MRI:
* **Đồng bộ hóa không gian**: Gọi hàm `resize` để đưa cả ảnh MRI và mặt nạ khối u về kích thước chuẩn định sẵn (`KICH_THUOC`).
* **Chuẩn hóa cường độ**: Gọi hàm `chuan_hoa` để quy đổi dải tín hiệu sáng về khoảng $[0, 255]$.
* **Lọc nhiễu linh hoạt**: Cho phép người dùng tùy chọn phương pháp lọc (mặc định là `median` hoặc `gauss`) thông qua tham số `cach_loc`.
* **Tăng cường tương phản tùy chọn**: Bổ sung cờ `them_histogram` (True/False) để kích hoạt quá trình cân bằng lược đồ xám khi cần thiết, giúp tối ưu hóa chất lượng đầu vào trước khi huấn luyện mô hình học sâu hoặc chạy thuật toán baseline Otsu.

In [8]:
def tien_xu_ly(anh, mask, cach_loc='median', them_histogram=False):
    anh, mask = resize(anh, mask)
    anh = chuan_hoa(anh)
    if cach_loc == 'median':
        anh = median(anh)
    elif cach_loc == 'gauss':
        anh = gauss(anh)
    else:
        print('Khong co cach loc nay:', cach_loc)

    if them_histogram:
        anh = can_bang_histogram(anh)

    return anh, mask

## 1.8 Các Hàm Tăng Cường Dữ Liệu Hình Học (Data Augmentation)
Nhóm hàm này thực hiện các biến đổi hình học ngẫu nhiên nhằm mở rộng không gian mẫu, giúp mô hình học sâu học được tính bất biến (invariance) và chống hiện tượng quá khớp (overfitting):
* **`lat_ngang` & `lat_doc` (`cv2.flip`)**: Thực hiện lật ảnh qua trục dọc hoặc trục ngang (mã `1` và `0`). Cả ảnh MRI và mặt nạ khối u đều được lật đồng thời để đảm bảo nhãn phân đoạn luôn khớp chính xác với vị trí mới của tổn thương.
* **`xoay` (`cv2.getRotationMatrix2D` & `cv2.warpAffine`)**: Xoay ảnh quanh tâm (`tam`) theo một góc ngẫu nhiên. Tương tự như bước tiền xử lý, hàm sử dụng nội suy tuyến tính (`INTER_LINEAR`) cho ảnh MRI để giữ độ mượt mà, và nội suy lân cận gần nhất (`INTER_NEAREST`) cho mặt nạ khối u nhằm bảo toàn tuyệt đối tính nhị phân của nhãn.

In [9]:

def lat_ngang(anh, mask):
    # 1 = lật trái <-> phải
    anh = cv2.flip(anh, 1)
    mask = cv2.flip(mask, 1)
    return anh, mask

def lat_doc(anh, mask):
    # 0 = lật trên <-> dưới
    anh = cv2.flip(anh, 0)
    mask = cv2.flip(mask, 0)
    return anh, mask

def xoay(anh, mask, goc):
    # Bước 1: tạo công thức xoay
    tam = (KICH_THUOC / 2, KICH_THUOC / 2)       # xoay quanh tâm ảnh
    ti_le = 1                                    # 1 = giữ nguyên kích thước
    ma_tran_xoay = cv2.getRotationMatrix2D(tam, goc, ti_le)

    # Bước 2: áp dụng công thức xoay cho ảnh và mask
    kich_thuoc_moi = (KICH_THUOC, KICH_THUOC)
    anh = cv2.warpAffine(anh, ma_tran_xoay, kich_thuoc_moi, flags=cv2.INTER_LINEAR)
    mask = cv2.warpAffine(mask, ma_tran_xoay, kich_thuoc_moi, flags=cv2.INTER_NEAREST)
    return anh, mask

## 1.9 Hàm Quản lý Quy Trình Tăng Cường Dữ Liệu Tổng Hợp (`augmentation`)
Hàm này đóng vai trò là bộ điều phối trung tâm cho các chiến lược tăng cường dữ liệu:
* **Điều kiện rẽ nhánh linh hoạt**: Nhận vào tham số `cach` (`'lat_ngang'`, `'lat_doc'`, hoặc `'xoay'`) cùng góc xoay tùy chỉnh (`goc`, mặc định là $10^\circ$) để gọi đúng hàm biến đổi hình học tương ứng.
* **Xử lý an toàn**: Tích hợp khối điều kiện kiểm tra lỗi (với thông báo cảnh báo khi tên phương thức tăng cường không hợp lệ), giúp pipeline chạy ổn định trong suốt quá trình huấn luyện mô hình U-Net.

In [10]:
def augmentation(anh, mask, cach, goc=10):
    if cach == 'lat_ngang':
        anh, mask = lat_ngang(anh, mask)
    elif cach == 'lat_doc':
        anh, mask = lat_doc(anh, mask)
    elif cach == 'xoay':
        anh, mask = xoay(anh, mask, goc)
    else:
        print('Khong co cach nay:', cach)

    return anh, mask

## 1.10 Hàm Tăng Cường Dữ Liệu Ngẫu Nhiên (`augmentation_ngau_nhien`)
Hàm này tích hợp tính ngẫu nhiên vào quy trình tiền xử lý của tập huấn luyện (Train) nhằm tối đa hóa khả năng tổng quát hóa của mô hình U-Net:
* **Xác suất lật ngẫu nhiên (50%)**: Sử dụng hàm `random.random()` với ngưỡng $0.5$ để quyết định độc lập việc áp dụng các phép lật ngang hoặc lật dọc, giúp mô hình không bị phụ thuộc vào hướng giải phẫu cố định.
* **Biến thiên góc xoay liên tục**: Sử dụng hàm `random.uniform()` để sinh ngẫu nhiên một góc xoay mượt mà trong khoảng giới hạn $[-\text{GOC\_XOAY\_TOI\_DA}, \text{GOC\_XOAY\_TOI\_DA}]$ (tương ứng $\pm 15^\circ$).
* **Đồng bộ hóa không gian**: Đảm bảo toàn bộ các phép biến đổi hình học ngẫu nhiên được áp dụng đồng thời và chính xác cho cả ảnh MRI lẫn mặt nạ khối u, mô phỏng chân thực các biến động về tư thế chụp trong thực tế lâm sàng.

In [11]:
def augmentation_ngau_nhien(anh, mask):

    if random.random() < 0.5:
        anh, mask = augmentation(anh, mask, 'lat_ngang')
    if random.random() < 0.5:
        anh, mask = augmentation(anh, mask, 'lat_doc')

    # Luôn xoay một góc nhỏ ngẫu nhiên từ -GOC_XOAY_TOI_DA đến GOC_XOAY_TOI_DA
    goc = random.uniform(-GOC_XOAY_TOI_DA, GOC_XOAY_TOI_DA)
    anh, mask = augmentation(anh, mask, 'xoay', goc)
    return anh, mask

## 1.11 Hàm Lập Danh Sách Đường Dẫn Tệp Dữ Liệu (`lay_danh_sach_file`)
Hàm này thực hiện việc tự động quét và tổng hợp toàn bộ đường dẫn của các tệp dữ liệu MRI trong thư mục nguồn:
* **Tạo chuỗi định danh tuần tự**: Duyệt qua vòng lặp từ $1$ đến `SO_FILE` ($3064$), kết hợp với phần mở rộng `.mat` để tái tạo chính xác tên của từng tệp dữ liệu (`1.mat`, `2.mat`, ..., `3064.mat`).
* **Quản lý đường dẫn đa nền tảng**: Sử dụng phương thức `os.path.join()` để ghép nối thư mục gốc với tên tệp một cách an toàn, độc lập với hệ điều hành (Windows, Linux, hay môi trường Kaggle Notebook).
* **Chuẩn bị cấu trúc đầu vào**: Trả về một danh sách (list) chứa toàn bộ các đường dẫn tuyệt đối, làm cơ sở dữ liệu đầu vào cho bước chia tách tập huấn luyện và khởi tạo Dataset PyTorch sau này.

In [12]:
def lay_danh_sach_file(thu_muc):
    # Tạo đường dẫn theo thứ tự 1.mat, 2.mat, ..., 3064.mat
    danh_sach_file = []
    for so in range(1, SO_FILE + 1):  
        ten = str(so) + '.mat'
        duong_dan = os.path.join(thu_muc, ten)
        danh_sach_file = danh_sach_file + [duong_dan]
    return danh_sach_file

## 1.12 Chiến Lược Phân Chia Dữ Liệu Theo Bệnh Nhân (`gom_theo_benh_nhan` & `chia_benh_nhan`)
Nhóm hàm này đóng vai trò quyết định trong việc thiết kế phương pháp thực nghiệm khoa học, ngăn chặn triệt để hiện tượng rò rỉ dữ liệu (data leakage):
* **`gom_theo_benh_nhan` (Lập từ điển nhóm theo PID)**: Duyệt qua toàn bộ tệp dữ liệu, trích xuất mã định danh bệnh nhân (`pid`) để nhóm các lát cắt MRI của cùng một người vào chung một danh sách. Điều này phản ánh chính xác thực tế lâm sàng khi một bệnh nhân có nhiều lát cắt chụp khác nhau.
* **`chia_benh_nhan` (Phân chia cấp độ bệnh nhân)**: 
  * Xáo trộn ngẫu nhiên danh sách mã bệnh nhân dựa trên `HAT_GIONG` đã cố định, đảm bảo tính khách quan và lặp lại.
  * Cắt danh sách bệnh nhân theo các tỷ lệ đã định hình (`TI_LE_TRAIN = 0.7`, `TI_LE_VAL = 0.15`, phần còn lại cho Test). 
  * **Ý nghĩa cốt lõi**: Việc chia tách theo *bệnh nhân* thay vì chia ngẫu nhiên từng *lát cắt ảnh* đảm bảo rằng các lát cắt thuộc cùng một khối u của một bệnh nhân sẽ không bị phân rải vừa ở tập Train vừa ở tập Test, giúp mô hình U-Net được đánh giá trên dữ liệu hoàn toàn chưa từng thấy (unseen patients).

In [13]:
def gom_theo_benh_nhan(danh_sach_file):
    # Kết quả: { mã bệnh nhân: [các file của bệnh nhân đó] }
    nhom = {}
    for duong_dan in danh_sach_file:
        pid = doc_ma_benh_nhan(duong_dan)
        if pid not in nhom:                 # bệnh nhân mới -> tạo list rỗng
            nhom[pid] = []
        nhom[pid] = nhom[pid] + [duong_dan] # thêm file vào list của bệnh nhân
    return nhom

def chia_benh_nhan(nhom):
    # Lấy danh sách mã bệnh nhân từ dict
    danh_sach_benh_nhan = []
    for pid in nhom.keys():
        danh_sach_benh_nhan = danh_sach_benh_nhan + [pid]

    # Xáo trộn danh sách bệnh nhân
#   random.seed ảnh hưởng đến TOÀN BỘ random của chương trình,
    # nên các lần augmentation sau đó cũng sẽ theo một trình tự cố định
    random.seed(HAT_GIONG)
    random.shuffle(danh_sach_benh_nhan)

    # Tính vị trí cắt: [0 ... het_train) là Train, [het_train ... het_val) là Val, còn lại là Test
    so_benh_nhan = len(danh_sach_benh_nhan)
    het_train = int(so_benh_nhan * TI_LE_TRAIN)
    so_benh_nhan_val = int(so_benh_nhan * TI_LE_VAL)
    het_val = het_train + so_benh_nhan_val

    benh_nhan_train = danh_sach_benh_nhan[:het_train]
    benh_nhan_val = danh_sach_benh_nhan[het_train:het_val]
    benh_nhan_test = danh_sach_benh_nhan[het_val:]
    return benh_nhan_train, benh_nhan_val, benh_nhan_test

## 1.13 Trực quan hóa Dữ liệu Gốc Ban Đầu (`Visualizing Raw Data`)
Đoạn mã này thực hiện việc kiểm tra trực quan trực tiếp trên tập dữ liệu thô trước khi đưa vào các bước tiền xử lý:
* **Lọc đại diện theo bệnh nhân**: Lấy ra tối đa $10$ bệnh nhân đầu tiên từ từ điển nhóm, chọn ra lát cắt đầu tiên (`nhom[pid][0]`) của mỗi bệnh nhân để hiển thị, đảm bảo tính đa dạng mẫu.
* **Hiển thị trực quan song song (`Matplotlib`)**:
  * **Hình bên trái**: Hiển thị ảnh cường độ xám (`cmap='gray'`) của lát cắt MRI gốc.
  * **Hình bên phải**: Chồng lớp viền mặt nạ khối u gốc (`plt.contour(..., colors='red')`) lên trên nền ảnh MRI, giúp kiểm tra trực quan sự chính xác của vị trí nhãn vùng tổn thương so với giải phẫu thực tế.

In [18]:
# ===== XEM ẢNH GỐC CỦA 10 BỆNH NHÂN ĐẦU TIÊN =====
# Lấy danh sách tất cả mã bệnh nhân từ từ điển 'nhom'
danh_sach_pid = list(nhom.keys())

# Chọn ra tối đa 10 bệnh nhân đầu tiên (hoặc ít hơn nếu tổng số bệnh nhân chưa đủ 10)
so_luong_can_xem = min(10, len(danh_sach_pid))

print(f"Đang hiển thị ảnh gốc của {so_luong_can_xem} bệnh nhân đầu tiên:")

for i in range(so_luong_can_xem):
    pid = danh_sach_pid[i]
    # Lấy file đầu tiên thuộc quyền sở hữu của bệnh nhân này
    duong_dan_file = nhom[pid][0]
    
    # Đọc ảnh gốc và mask từ file .mat
    anh_goc, mask_goc = doc_file_mat(duong_dan_file)
    
    # Hiển thị bằng matplotlib
    plt.figure(figsize=(6, 3))
    
    # Hình 1: Ảnh gốc MRI
    plt.subplot(1, 2, 1)
    plt.imshow(anh_goc, cmap='gray')
    plt.title(f'Benh nhan {i+1} (PID gốc)')
    
    # Hình 2: Viền khối u gốc
    plt.subplot(1, 2, 2)
    plt.imshow(anh_goc, cmap='gray')
    plt.contour(mask_goc, colors='red')
    plt.title('Tumor Mask goc')
    
    plt.show()

NameError: name 'nhom' is not defined

## 1.14 Hàm Ánh Xạ Danh Sách Bệnh Nhân Thành Danh Sách Tệp (`lay_file_cua_benh_nhan`)
Hàm này thực hiện công đoạn chuyển đổi ngược từ cấp độ định danh bệnh nhân sang cấp độ tệp dữ liệu chi tiết:
* **Tổng hợp tệp theo nhóm**: Duyệt qua từng mã bệnh nhân (`pid`) có trong danh sách phân chia (Train, Val, hoặc Test), truy vấn vào từ điển nhóm (`nhom`) để lấy toàn bộ các đường dẫn lát cắt MRI thuộc quyền sở hữu của bệnh nhân đó.
* **Tạo danh sách hoàn chỉnh**: Gom nối các đường dẫn lại thành một danh sách tệp thống nhất, phục vụ trực tiếp cho việc khởi tạo các đối tượng `Dataset` và `DataLoader` của PyTorch trong các bước tiếp theo.

In [15]:
def lay_file_cua_benh_nhan(danh_sach_benh_nhan, nhom):
    # Đổi danh sách bệnh nhân -> danh sách file của các bệnh nhân đó
    danh_sach_file = []
    for pid in danh_sach_benh_nhan:
        danh_sach_file = danh_sach_file + nhom[pid]
    return danh_sach_file


## 1.15 Hàm Kiểm Tra Trực Quan Tác Động Tiền Xử Lý (`xem_anh`)
Hàm này cung cấp công cụ trực quan hóa toàn diện nhằm đánh giá trực tiếp chất lượng của đường ống tiền xử lý trên một lát cắt MRI bất kỳ:
* **Truy xuất và xử lý đồng bộ**: Nhận vào số thứ tự tệp (`so_file`), tự động định vị đường dẫn, đọc dữ liệu gốc và thực hiện toàn bộ quy trình tiền xử lý (`tien_xu_ly`).
* **Hiển thị đa chiều song song (`1 dòng x 3 cột`)**:
  * **Hình 1 (`Anh goc`)**: Hiển thị ảnh MRI thô ban đầu để làm đối chứng.
  * **Hình 2 (`Sau tien xu ly`)**: Hiển thị ảnh sau khi đã resize, chuẩn hóa cường độ sáng và lọc nhiễu, giúp quan sát độ sắc nét và độ tương phản cải thiện.
  * **Hình 3 (`Vien khoi u`)**: Chồng lớp mặt nạ nhãn khối u (`mask_xu_ly`) lên trên ảnh đã xử lý dưới dạng đường viền đỏ (`plt.contour`), cho phép kiểm tra xem các bước biến đổi không gian có làm lệch hoặc biến dạng ranh giới khối u hay không.

In [16]:
def xem_anh(so_file):
    # Ghép đường dẫn, ví dụ so_file = 1 -> data\1.mat
    ten = str(so_file) + '.mat'
    duong_dan = os.path.join(THU_MUC_DU_LIEU, ten)

    # Đọc ảnh gốc và ảnh sau tiền xử lý
    anh_goc, mask_goc = doc_file_mat(duong_dan)
    anh_xu_ly, mask_xu_ly = tien_xu_ly(anh_goc, mask_goc)

    # Hình 1: ảnh gốc
    plt.subplot(1, 3, 1)
    plt.imshow(anh_goc, cmap='gray')
    plt.title('Anh goc')

    # Hình 2: ảnh sau tiền xử lý
    plt.subplot(1, 3, 2)
    plt.imshow(anh_xu_ly, cmap='gray')
    plt.title('Sau tien xu ly')

    # Hình 3: ảnh sau tiền xử lý + viền khối u màu đỏ
    plt.subplot(1, 3, 3)
    plt.imshow(anh_xu_ly, cmap='gray')
    plt.contour(mask_xu_ly, colors='red')
    plt.title('Vien khoi u')

    plt.show()


## 1.16 Xây Dựng Lớp Tùy Chỉnh Quản Lý Dữ Liệu PyTorch (`MRIDataset`)
Lớp này kế thừa từ `torch.utils.data.Dataset` chuẩn của PyTorch, chịu trách nhiệm đóng gói và cung cấp dữ liệu đầu vào cho quá trình huấn luyện và đánh giá mô hình:
* **Hàm khởi tạo (`__init__`)**: Nhận vào danh sách tệp (`danh_sach_file`) và cờ định danh `train` (nếu là `True` sẽ kích hoạt quy trình tăng cường dữ liệu ngẫu nhiên).
* **Hàm lấy kích thước (`__len__`)**: Trả về tổng số lượng lát cắt MRI có trong tập dữ liệu hiện tại, giúp `DataLoader` quản lý các batch dữ liệu.
* **Hàm nạp và xử lý từng phần tử (`__getitem__`)**:
  * Đọc file `.mat`, chạy qua pipeline tiền xử lý chung (`tien_xu_ly`).
  * Kiểm tra điều kiện `self.train`: nếu đúng, áp dụng các phép tăng cường dữ liệu ngẫu nhiên (`augmentation_ngau_nhien`) để nâng cao khả năng học tập của mô hình.
  * **Chuẩn hóa cường độ & Kiểu dữ liệu**: Chuyển đổi ma trận ảnh từ thang $[0, 255]$ về khoảng chuẩn hóa $[0, 1]$ dưới dạng số thực `float32`.
  * **Định hình lại không gian kênh (`reshape`)**: Thêm chiều kênh đơn (`1`) cho ảnh xám và mặt nạ, chuyển từ kích thước $(256, 256)$ thành dạng chuẩn của mạng nơ-ron $(1, 256, 256)$.
  * **Chuyển đổi sang PyTorch Tensor**: Ép kiểu dữ liệu từ mảng `NumPy` sang `torch.Tensor` để tương thích hoàn hảo với các tầng tính toán tensor và GPU của PyTorch.

In [19]:
class MRIDataset(Dataset):
    def __init__(self, danh_sach_file, train=False):
        self.danh_sach_file = danh_sach_file    # list đường dẫn file của tập này
        self.train = train                      # True = tập Train (có augmentation)

    def __len__(self):
        return len(self.danh_sach_file)

    def __getitem__(self, i):
        anh, mask = doc_file_mat(self.danh_sach_file[i])
        anh, mask = tien_xu_ly(anh, mask)

        if self.train:                          # chỉ tập Train mới augmentation
            anh, mask = augmentation_ngau_nhien(anh, mask)

        # đổi 0 -> 255 thành 0 -> 1 để dễ train  
        anh = anh.astype(np.float32)
        anh = anh / 255.0
        mask = mask.astype(np.float32)

        # Thêm chiều "kênh": (256, 256) -> (1, 256, 256), vì ảnh xám chỉ có 1 kênh
        anh = anh.reshape(1, KICH_THUOC, KICH_THUOC)
        mask = mask.reshape(1, KICH_THUOC, KICH_THUOC)

        # Đổi từ numpy sang dạng tensor của PyTorch
        anh = torch.from_numpy(anh)
        mask = torch.from_numpy(mask)
        return anh, mask


## 1.17 Khối Thực Thi Chính & Kiểm Tra Quy Trình Dữ Liệu (`Main Execution Pipeline`)
Đoạn mã này đóng vai trò là điểm khởi chạy toàn bộ quy trình chuẩn bị dữ liệu, tích hợp tất cả các thành phần đã xây dựng từ đầu đến cuối:
* **Phân chia tập dữ liệu cấp độ bệnh nhân (Phần 1)**: 
  * Quét thư mục để lấy toàn bộ danh sách tệp, nhóm các lát cắt theo mã định danh bệnh nhân (`pid`) nhằm tránh triệt để hiện tượng rò rỉ dữ liệu (data leakage).
  * Thực hiện chia tách danh sách bệnh nhân thành 3 tập riêng biệt (Train, Validation, Test) theo tỷ lệ định sẵn, sau đó quy đổi ngược lại thành danh sách các tệp chi tiết tương ứng.
* **Khởi tạo các đối tượng `MRIDataset` (Phần 2)**:
  * Khởi tạo 3 đối tượng dataset tương ứng cho từng tập, trong đó cờ `train=True` chỉ được kích hoạt riêng cho tập huấn luyện nhằm áp dụng các phép tăng cường dữ liệu ngẫu nhiên.
  * In kiểm tra nhanh kích thước, kiểu dữ liệu (`dtype`) và khoảng giá trị cường độ sáng sau khi chuẩn hóa (`[0.0, 1.0]`) của một mẫu tensor bất kỳ.
* **Đóng gói dữ liệu hàng loạt với `DataLoader` (Phần 3)**:
  * Khởi tạo `DataLoader` với kích thước lô (`batch_size = 8`) cùng cơ chế xáo trộn (`shuffle=True`) cho tập huấn luyện.
  * Trích xuất thử nghiệm một lô dữ liệu đầu tiên (`lo_anh`, `lo_mask`) để kiểm tra tính toàn vẹn của chiều tensor đầu vào (ví dụ: `torch.Size([8, 1, 256, 256])`), sẵn sàng bàn giao cho bước xây dựng và huấn luyện mô hình mạng nơ-ron U-Net tiếp theo.

In [ ]:
if __name__ == "__main__":
    # 1. Chia dữ liệu theo bệnh nhân
    danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
    nhom = gom_theo_benh_nhan(danh_sach_file)
    benh_nhan_train, benh_nhan_val, benh_nhan_test = chia_benh_nhan(nhom)
    print("So benh nhan:", len(benh_nhan_train), len(benh_nhan_val), len(benh_nhan_test))

    file_train = lay_file_cua_benh_nhan(benh_nhan_train, nhom)
    file_val = lay_file_cua_benh_nhan(benh_nhan_val, nhom)
    file_test = lay_file_cua_benh_nhan(benh_nhan_test, nhom)
    print("So file:", len(file_train), len(file_val), len(file_test))

    # 2. Tạo 3 Dataset, chỉ Train có augmentation
    tap_train = MRIDataset(file_train, train=True)
    tap_val = MRIDataset(file_val)
    tap_test = MRIDataset(file_test)

    anh, mask = tap_train[0]
    gia_tri_nho_nhat = anh.min().item()     # .item() lấy con số ra khỏi tensor
    gia_tri_lon_nhat = anh.max().item()
    print("1 anh:", anh.shape, anh.dtype, gia_tri_nho_nhat, gia_tri_lon_nhat)

    # 3. DataLoader: gom 8 ảnh thành 1 lô để đưa vào mô hình
    # Chỉ lấy lô đầu tiên để xem thử
    bo_nap = DataLoader(tap_train, batch_size=8, shuffle=True)
    for lo_anh, lo_mask in bo_nap:
        print("1 lo:", lo_anh.shape, lo_mask.shape)
        break

## 1.18 Trực quan hóa Kiểm tra Tiền Xử Lý Hàng Loạt trên 10 Bệnh Nhân Đầu Tiên
Đoạn mã này cung cấp một cái nhìn tổng quan và hệ thống để kiểm tra chất lượng của toàn bộ quy trình tiền xử lý trên các mẫu thực tế:
* **Ánh xạ từ mã bệnh nhân đến tệp dữ liệu**: Duyệt qua danh sách các mã định danh (`pid`), tự động trích xuất tệp đại diện đầu tiên của từng bệnh nhân.
* **Trích xuất số thứ tự tệp động**: Sử dụng `os.path.basename` và phương thức cắt chuỗi (`split('.')`) để lấy chính xác số thứ tự tệp (`so_file`) từ đường dẫn, đảm bảo tương thích hoàn hảo với hàm `xem_anh` đã xây dựng.
* **Hiển thị trực quan song song**: In thông tin chi tiết (STT, PID, Số file) ra màn hình console và gọi hàm `xem_anh` để xuất ra chuỗi 3 biểu đồ trực quan (ảnh gốc, ảnh sau tiền xử lý, và ảnh chèn viền khối u đỏ) cho từng bệnh nhân, giúp nhanh chóng phát hiện các bất thường (nếu có) trước khi bước vào giai đoạn huấn luyện mô hình học sâu.

In [ ]:
# ===== XEM ẢNH CỦA 10 BỆNH NHÂN ĐẦU TIÊN =====
# 1. Lấy danh sách tất cả các mã bệnh nhân (PID) từ từ điển 'nhom'
danh_sach_pid = list(nhom.keys())

# 2. Chọn số lượng cần xem (tối đa 10 bệnh nhân)
so_luong_can_xem = min(10, len(danh_sach_pid))
print(f"Đang hiển thị kết quả của {so_luong_can_xem} bệnh nhân đầu tiên:")

# 3. Vòng lặp duyệt qua từng bệnh nhân
for i in range(so_luong_can_xem):
    pid = danh_sach_pid[i]
    
    # Lấy file đầu tiên thuộc quyền sở hữu của bệnh nhân này để làm đại diện
    duong_dan_file = nhom[pid][0]
    
    # Tách lấy số thứ tự của file từ đường dẫn để truyền vào hàm xem_anh của bạn
    # (Ví dụ đường dẫn là 'data/55.mat' thì lấy số '55')
    ten_file_chi_tiet = os.path.basename(duong_dan_file) # Lấy tên file '55.mat'
    so_file = int(ten_file_chi_tiet.split('.')[0])        # Lấy số 55
    
    print(f"--- Đang xem bệnh nhân thứ {i+1} (Mã PID: {pid}, File số: {so_file}) ---")
    
    # Gọi lại hàm xem ảnh có sẵn của bạn
    xem_anh(so_file)

# 2.Modeling & Training (Dựng & Huấn luyện U-Net)


## 2.1 Xây Dựng Phương Pháp Đối Chứng Truyền Thống: Ngưỡng Otsu (`phuong_phap_otsu_doi_chung`)
Hàm này cài đặt thuật toán phân đoạn ngưỡng Otsu tự động nhằm đóng vai trò là phương pháp cơ sở (baseline) truyền thống, phục vụ cho việc đánh giá và so sánh hiệu năng với mô hình học sâu U-Net:
* **Chuẩn bị dữ liệu đầu vào**: Tải tệp dữ liệu MRI, thực hiện đồng bộ kích thước (`resize`) và chuẩn hóa cường độ sáng (`chuan_hoa`) để đảm bảo tính nhất quán cho không gian ảnh xám.
* **Phân ngưỡng tự động (`cv2.threshold` kết hợp `cv2.THRESH_OTSU`)**: Thuật toán tự động phân tích lược đồ xám (histogram) để tìm ra giá trị ngưỡng cường độ sáng tối ưu (`nguong_otsu`) bằng cách tối đa hóa phương sai giữa các lớp (inter-class variance), loại bỏ sự phụ thuộc vào việc chọn ngưỡng thủ công.
* **Trực quan hóa đối chứng trực diện (`1 dòng x 3 cột`)**: 
  * **Hình 1 (`Ảnh MRI đầu vào`)**: Ảnh sau khi đã qua quy trình tiền xử lý chuẩn.
  * **Hình 2 (`Kết quả Otsu`)**: Vùng tổn thương được phân đoạn tự động bởi thuật toán Otsu kèm theo giá trị ngưỡng tìm được.
  * **Hình 3 (`Mask Thực Tế - Ground Truth`)**: Nhãn chuẩn từ chuyên gia y tế, cho phép quan sát trực quan những hạn chế của phương pháp truyền thống (như độ nhiễu cao, nhạy cảm với cường độ sáng) khi phân đoạn các cấu trúc y tế phức tạp.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath('.'))
# Import trực tiếp từ file dataset.py của bạn


def phuong_phap_otsu_doi_chung(so_file):
    ten = str(so_file) + '.mat'
    duong_dan = os.path.join(THU_MUC_DU_LIEU, ten)
    anh_goc, mask_goc = doc_file_mat(duong_dan)
    
    anh_xu_ly, _ = resize(anh_goc, mask_goc)
    anh_xu_ly = chuan_hoa(anh_xu_ly)
    
    nguong_otsu, anh_otsu = cv2.threshold(anh_xu_ly, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print(f"Ngưỡng tối ưu tìm được bằng Otsu: {nguong_otsu}")

    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.imshow(anh_xu_ly, cmap='gray')
    plt.title('Ảnh MRI đầu vào')

    plt.subplot(1, 3, 2)
    plt.imshow(anh_otsu, cmap='gray')
    plt.title(f'Kết quả Otsu (Ngưỡng: {nguong_otsu})')

    plt.subplot(1, 3, 3)
    plt.imshow(mask_goc, cmap='gray')
    plt.title('Mask Thực Tế (Ground Truth)')

    plt.show()


## 2.2 Khối Thực Thi Thử Nghiệm Phương Pháp Ngưỡng Otsu Hàng Loạt (`Otsu Baseline Execution`)
Đoạn mã này thực thi việc đánh giá thực nghiệm phương pháp phân đoạn truyền thống Otsu trên tập dữ liệu thực tế:
* **Truy xuất định danh thông minh**: Tự động quét từ điển nhóm bệnh nhân, trích xuất mã định danh và lấy tệp đại diện đầu tiên của tối đa $10$ bệnh nhân đầu tiên.
* **Trích xuất số thứ tự tệp tự động**: Xử lý chuỗi đường dẫn thông qua `os.path.basename` để lấy chính xác số thứ tự tệp, đảm bảo tính liền mạch khi gọi hàm phân đoạn đối chứng.
* **Đánh giá trực quan đối chiếu**: In nhật ký tiến trình (console log) chi tiết cho từng bệnh nhân và kích hoạt hàm `phuong_phap_otsu_doi_chung`, xuất ra các biểu đồ so sánh trực quan giữa kết quả phân đoạn bằng ngưỡng Otsu tự động và mặt nạ chuẩn y tế (`Ground Truth`), qua đó làm nổi bật cơ sở khoa học và lý do cần phải ứng dụng kiến trúc học sâu U-Net chuyên sâu.

In [ ]:
if __name__ == '__main__':
    # 1. Lấy danh sách toàn bộ file hoặc danh sách bệnh nhân từ nhóm
    danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
    nhom = gom_theo_benh_nhan(danh_sach_file)
    danh_sach_pid = list(nhom.keys())
    
    # 2. Xác định số lượng cần chạy (tối đa 10 bệnh nhân đầu tiên)
    so_luong_can_chay = min(10, len(danh_sach_pid))
    print(f"Đang chạy phương pháp Otsu đối chứng cho {so_luong_can_chay} bệnh nhân đầu tiên:")

    # 3. Vòng lặp duyệt qua từng bệnh nhân
    for i in range(so_luong_can_chay):
        pid = danh_sach_pid[i]
        
        # Lấy file đại diện đầu tiên của bệnh nhân đó
        duong_dan_file = nhom[pid][0]
        
        # Trích xuất số file từ tên đường dẫn (ví dụ 'data/55.mat' -> lấy số 55)
        ten_file_chi_tiet = os.path.basename(duong_dan_file)
        so_file = int(ten_file_chi_tiet.split('.')[0])
        
        print(f"\n--- Đang xử lý bệnh nhân thứ {i+1} (File số: {so_file}) ---")
        
        # Gọi hàm phương pháp otsu đối chứng cho file này
        phuong_phap_otsu_doi_chung(so_file)

## 2.3 Xây Dựng Kiến Trúc Mạng Nơ-ron Học Sâu U-Net & Kiểm Tra Số Lượng Tham Số
Đoạn mã này hiện thực hóa kiến trúc mạng U-Net kinh điển dành cho bài toán phân đoạn ảnh y tế (Medical Image Segmentation), kết hợp với tiện ích kiểm tra quy mô mô hình:
* **Khối Cơ Bản `DoubleConv`**: Định nghĩa khối tích chconvolution kép tiêu chuẩn bao gồm hai tầng `Conv2d` (kích thước nhân $3 \times 3$, đệm lề `padding=1` để giữ nguyên kích thước không gian), kết hợp chuẩn hóa hàng loạt (`BatchNorm2d`) và hàm kích hoạt phi tuyến tính `ReLU` (với chế độ `inplace=True` tối ưu hóa bộ nhớ GPU).
* **Kiến trúc Tổng Thể `UNet`**:
  * **Đường dẫn thu nhỏ (Encoder / Downsampling)**: Sử dụng các lớp `MaxPool2d` kết hợp `DoubleConv` để trích xuất đặc trưng ngữ cảnh từ chi tiết cục bộ đến toàn cục qua các tầng độ sâu (từ $1$ kênh đầu vào ảnh MRI xám lên đến $512$ kênh đặc trưng).
  * **Đường dẫn mở rộng (Decoder / Upsampling)**: Sử dụng `ConvTranspose2d` để khôi phục lại kích thước không gian, đồng thời áp dụng cơ chế kết nối tắt (`torch.cat`) để ghép nối các bản đồ đặc trưng từ phần encoder tương ứng qua từng mức độ (skip connections), giúp khôi phục chi tiết biên sắc nét của khối u.
  * **Tầng xuất lượng (`outc`)**: Sử dụng phép tích chập $1 \times 1$ để ánh xạ các đặc trưng bậc cao về số lượng kênh nhãn đầu ra (`out_channels=1`), tạo ra các logits phân đoạn nhị phân.
* **Hàm Thống Kê Tham Số `count_parameters`**: 
  * Duyệt qua toàn bộ các ten-sơ trọng số trong mô hình, sử dụng phương thức `.numel()` để tính tổng số lượng tham số (`total_params`) và số lượng tham số cần tối ưu hóa (`trainable_params`).
  * In ra màn hình console báo cáo chi tiết quy mô của mạng U-Net, giúp kiểm soát tài nguyên tính toán trước khi bước vào vòng lặp huấn luyện (Training Loop).

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()
        
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 512))

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(768, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(384, 128)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(192, 64)
        
        self.up4 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv2(x)

        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv3(x)

        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv4(x)

        logits = self.outc(x)
        return logits
def count_parameters(model):
    # Tổng số tham số của mô hình
    total_params = sum(p.numel() for p in model.parameters())
    # Số tham số được cập nhật trong quá trình huấn luyện
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    return total_params, trainable_params

# Kiểm tra với mô hình UNet của bạn
model = UNet(in_channels=1, out_channels=1)
count_parameters(model)

## 2.4 Tối Ưu Hóa Hàm Tiền Xử Lý Ảnh Đơn Lẻ (`tien_xu_ly`)
Hàm này cung cấp quy trình tiền xử lý tiêu chuẩn, tối ưu hóa trực tiếp trên ảnh MRI đơn lẻ trước khi đưa vào mô hình học sâu:
* **Kiểm soát và chuyển đổi kiểu dữ liệu an toàn**: Kiểm tra định dạng ma trận đầu vào (`dtype`), tự động sử dụng hàm `cv2.normalize` để đưa dải cường độ sáng về chuẩn `uint8` (`[0, 255]`) trường hợp ảnh chưa được chuẩn hóa, đảm bảo tương thích hoàn hảo với các bộ lọc của thư viện OpenCV.
* **Khử nhiễu bảo toàn biên (`cv2.medianBlur`)**: Áp dụng bộ lọc trung vị với kích thước nhân hạt `kernel_size = 3`, giúp loại bỏ hiệu quả các loại nhiễu hạt y tế mà vẫn **giữ nguyên vẹn các đường biên sắc nét** của vùng ranh giới khối u.
* **Chuẩn hóa biên độ số thực (`Float32 Normalization`)**: Chuyển đổi và chia tỷ lệ cường độ điểm ảnh về đoạn $[0.0, 1.0]$ dưới dạng số thực `float32`, sẵn sàng tương thích hoàn hảo với các tầng xử lý tensor của mạng PyTorch.

In [ ]:
def tien_xu_ly(img, target_size=(256, 256)):
    """
    Hàm tiền xử lý ảnh MRI:
    1. Khử nhiễu bằng bộ lọc trung vị (Median Filter).
    2. Chuẩn hóa cường độ pixel về khoảng [0, 1].
    """
    # Đảm bảo ảnh ở định dạng float hoặc chuẩn bị cho lọc median
    if img.dtype != np.uint8:
        # Chuẩn hóa tạm về uint8 nếu cần lọc qua cv2, hoặc dùng scipy
        img_uint8 = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    else:
        img_uint8 = img

    # Áp dụng Median Filter để khử nhiễu hạt / nhiễu xung quanh nhưng vẫn giữ nguyên biên khối u
    filtered_img = cv2.medianBlur(img_uint8, kernel_size=3) # kernel_size thường là 3 hoặc 5

    # Chuẩn hóa lại về [0.0, 1.0] dạng float32 cho PyTorch
    normalized_img = filtered_img.astype(np.float32) / 255.0
    
    return normalized_img

## 2.5 Hoàn Thiện Lớp Quản Lý Dữ Liệu PyTorch Nâng Cao (`MRIDataset`)
Phiên bản cập nhật của lớp `MRIDataset` được tối ưu hóa chuyên biệt để nạp trực tiếp các cặp tệp ảnh MRI và mặt nạ tương ứng:
* **Nạp dữ liệu đa thành phần qua chỉ mục (`__getitem__`)**: Truy xuất đồng thời đường dẫn ảnh và mặt nạ (`path_anh`, `path_mask`) từ danh sách, sử dụng OpenCV (`cv2.imread`) để đọc dưới dạng ảnh xám một cách nhanh chóng.
* **Xử lý chuyên biệt cho Ảnh & Mặt nạ (Mask)**:
  * **Ảnh MRI**: Được chuyển qua hàm tiền xử lý chuẩn (`tien_xu_ly`) để khử nhiễu bằng bộ lọc trung vị và chuẩn hóa cường độ sáng.
  * **Mặt nạ phân đoạn**: Được thay đổi kích thước bằng phương pháp nội suy hàng xóm gần nhất (`cv2.INTER_NEAREST`) để bảo toàn nguyên vẹn nhãn nhị phân, sau đó nhị phân hóa về ngưỡng chuẩn và ép kiểu số thực (`[0.0, 1.0]`).
* **Định hình không gian Tensor chuẩn PyTorch**: Sử dụng phương thức `.unsqueeze(0)` để tự động thêm chiều kênh (`channel = 1`), đưa kích thước tensor đầu ra về đúng chuẩn cấu trúc `[C, H, W]` (`[1, 256, 256]`), đảm bảo sự tương thích tuyệt đối khi kết nối với các lô dữ liệu huấn luyện (`DataLoader`).

In [ ]:
from torch.utils.data import Dataset
import torch

class MRIDataset(Dataset):
    def __init__(self, danh_sach_file, train=True, target_size=(256, 256)):
        self.danh_sach_file = danh_sach_file
        self.train = train
        self.target_size = target_size

    def __len__(self):
        return len(self.danh_sach_file)

    def __getitem__(self, idx):
        # 1. Đọc ảnh và mask từ đường dẫn (tuỳ thuộc vào cấu trúc lưu trữ của bạn)
        path_anh, path_mask = self.danh_sach_file[idx]
        
        # Đọc ảnh xám (grayscale)
        anh = cv2.imread(path_anh, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(path_mask, cv2.IMREAD_GRAYSCALE)
        
        # 2. Áp dụng tiền xử lý chung (có median filter) cho ảnh
        anh = tien_xu_ly(anh, self.target_size)
        
        # Chuẩn hóa mask về [0, 1] (binary mask)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)

        # 3. Chuyển đổi sang PyTorch Tensor (Shape: [C, H, W])
        anh = torch.tensor(anh, dtype=torch.float32).unsqueeze(0)   # Thêm chiều channel [1, H, W]
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0) # Thêm chiều channel [1, H, W]

        return anh, masks if 'masks' in locals() else (anh, mask)

## 2.6 Khởi Tạo Từ Điển Theo Dõi Lịch Sử Huấn Luyện (`history`)
Đoạn mã này thiết lập cấu trúc dữ liệu dạng từ điển (`dict`) nhằm ghi nhận và lưu trữ toàn bộ các chỉ số hiệu năng trong suốt tiến trình huấn luyện mô hình U-Net:
* **`train_loss`**: Lưu lại giá trị hàm mất mát (loss) trung bình trên tập huấn luyện qua từng kỷ nguyên (epoch), giúp theo dõi tốc độ hội tụ của mô hình.
* **`val_loss`**: Ghi nhận giá trị hàm mất mát trên tập kiểm định (validation), đóng vai trò cốt lõi trong việc phát hiện hiện tượng học vẹt (overfitting).
* **Các chỉ số đánh giá chuyên sâu (`val_dice`, `val_iou`)**: Lưu trữ hệ số tương đồng Dice Coefficient và độ đo Intersection over Union (IoU) trên tập kiểm định, cung cấp tiêu chí định lượng khách quan và chính xác nhất cho bài toán phân đoạn vùng tổn thương u não.

In [ ]:
history = {
    'train_loss': [],
    'val_loss': [],
    'val_dice': [],
    'val_iou': []
}

## 2.7 Xây Dựng Vòng Lặp Huấn Luyện Toàn Diện Cho Mô Hình U-Net (`huan_luyen_mo_hinh`)
Hàm này hiện thực hóa toàn bộ quy trình tối ưu hóa, kiểm định và lưu trữ trọng số mô hình trong suốt quá trình huấn luyện:
* **Khởi tạo Môi trường & Thiết bị (`Device & Seed Setting`)**: 
  * Cố định toàn bộ các hạt giống ngẫu nhiên (`set_seed`) để đảm bảo tính tái lập của kết quả thực nghiệm.
  * Tự động phát hiện và cấu hình thiết bị tính toán (`GPU CUDA` hoặc `CPU`), kèm theo cảnh báo nếu chưa kích hoạt bộ gia tốc phần cứng trên nền tảng Kaggle.
* **Chuẩn bị Dữ liệu & Bộ Tối Ưu Hóa (`DataLoaders & Optimization`)**:
  * Khởi tạo `DataLoader` cho tập Train (có xáo trộn) và Validation (không xáo trộn) với tham số `num_workers=0` để đảm bảo ổn định đa tiến trình trên môi trường Kaggle.
  * Sử dụng hàm mất mát kết hợp `DiceBCELoss` (tổng hợp giữa Binary Cross-Entropy và Dice Loss) cùng thuật toán tối ưu `Adam` (tốc độ học khởi tạo `1e-3`).
  * Tích hợp bộ điều chỉnh tốc độ học động `ReduceLROnPlateau` để tự động giảm tốc độ học khi giá trị mất mát kiểm định (`val_loss`) chững lại.
* **Vòng Lặp Huấn Luyện Qua Các Kỷ Nguyên (`Training & Validation Loop`)**:
  * **Huấn luyện (`model.train()`)**: Lan truyền tiến, tính toán hàm mất mát, lan truyền ngược (`loss.backward()`) và cập nhật trọng số, đồng thời in tiến độ chi tiết theo từng cụm batch.
  * **Kiểm định (`model.eval()`)**: Đánh giá mô hình trên tập validation mà không cập nhật trọng số (`torch.no_grad()`), đồng thời tính toán các độ đo chuyên sâu (`Dice Coefficient` và `IoU`).
* **Cơ Chế Lưu Trọng Sốt Tốt Nhất & Lịch Sử (`Checkpointing & Logging`)**:
  * Theo dõi và tự động lưu các tệp trọng số tốt nhất dựa trên giá trị mất mát (`best_loss_model.pth`) và hệ số tương đồng (`best_dice_model.pth`).
  * Lưu toàn bộ bảng ghi lịch sử huấn luyện (`history`) ra tệp định dạng `training_history.json`, phục vụ cho việc vẽ biểu đồ phân tích hiệu năng mô hình ở các bước tiếp theo.

In [ ]:
import torch
import torch.optim as optim
import sys
import os
import json
from torch.utils.data import DataLoader

# Import từ file dataset.py của bạn
def tinh_dice_iou(dau_ra, tarets, threshold=0.5, smooth=1e-5):
    preds = (torch.sigmoid(dau_ra) > threshold).float()

    preds = preds.view(-1)
    targets = targets.view(-1)

    intersection = (preds * targets).sum()

    dice = (2. * intersection + smooth) / (preds.sum() + targets.sum() + smooth)

    union = preds.sum() + targets.sum() - intersection
    iou = (intersection + smooth) / (union + smooth)

    return dice.item(), iou.item()
class DiceBCELoss(torch.nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()
        self.bce = torch.nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1e-5):
        bce_loss = self.bce(inputs, targets)
        
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        intersection = (inputs * targets).sum()                            
        dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)  
        
        return bce_loss + dice_loss
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 
def huan_luyen_mo_hinh():
    set_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Đang chạy trên thiết bị:", device)
    
    if device.type == 'cpu':
        print("CẢNH BÁO: Bạn đang chạy bằng CPU! Hãy kiểm tra lại Accelerator trên Kaggle và chuyển sang GPU.")

    print("Đang chuẩn bị dữ liệu...")
    danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
    nhom = gom_theo_benh_nhan(danh_sach_file)
    benh_nhan_train, benh_nhan_val, _ = chia_benh_nhan(nhom)

    file_train = lay_file_cua_benh_nhan(benh_nhan_train, nhom)
    file_val = lay_file_cua_benh_nhan(benh_nhan_val, nhom)

    tap_train = MRIDataset(file_train, train=True)
    tap_val = MRIDataset(file_val, train=False)

    # 💡 QUAN TRỌNG: Đổi num_workers=0 để tránh lỗi treo h5py trên Kaggle
    train_loader = DataLoader(tap_train, batch_size=8, shuffle=True, num_workers=0)
    val_loader = DataLoader(tap_val, batch_size=8, shuffle=False, num_workers=0)

    model = UNet(in_channels=1, out_channels=1).to(device)
    criterion = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    
    
    so_epoch = 30 
    best_val_loss = float('inf')
    best_val_dice = 0.0

    patience_limit = 6
    patience_counter = 0

    print("Bắt đầu huấn luyện...")
    for epoch in range(so_epoch):
        model.train()
        train_loss = 0.0
        
        # Thêm biến đếm batch để in tiến độ
        tong_so_batch = len(train_loader)
        
        for batch_idx, (anhs, masks) in enumerate(train_loader):
            anhs, masks = anhs.to(device), masks.to(device)

            optimizer.zero_grad()
            dau_ra = model(anhs)
            loss = criterion(dau_ra, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * anhs.size(0)
            
            # In tiến độ mỗi 20 batch để bạn biết code vẫn đang chạy bình thường
            if (batch_idx + 1) % 20 == 0 or (batch_idx + 1) == tong_so_batch:
                print(f"Epoch [{epoch+1}/{so_epoch}] - Batch [{batch_idx+1}/{tong_so_batch}] - Loss: {loss.item():.4f}")

        train_loss = train_loss / len(tap_train)

        model.eval()
        val_loss = 0.0
        val_dice = 0.0
        val_iou = 0.0
        
        with torch.no_grad():
            for anhs, masks in val_loader:
                anhs, masks = anhs.to(device), masks.to(device)
                dau_ra = model(anhs)
                loss = criterion(dau_ra, masks)
                
                val_loss += loss.item() * anhs.size(0)

                batch_dice, batch_iou = tinh_dice_iou(dau_ra, masks)
                val_dice += batch_dice * anhs.size(0)
                val_iou +=  batch_iou * anhs.size(0)

        val_loss = val_loss / len(tap_val)
        val_dice = val_dice / len(tap_val)
        val_iou = val_iou / len(tap_val)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        history['val_iou'].append(val_iou)
        
        print(f"==> KẾT QUẢ EPOCH [{epoch+1}/{so_epoch}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f}")

        scheduler.step(val_loss)
        

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_loss_model.pth')
            print("--> Đã lưu model tốt nhất vào best_loss_model.pth!")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), 'best_dice_model.pth')
            print("--> Đã lưu model tốt nhất vào best_dice_model.pth")
    # Lưu toàn bộ history ra file json
    with open('training_history.json', 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=4)
    print("Đã lưu lịch sử huấn luyện vào file training_history.json thành công!")
    
if __name__ == '__main__':
    huan_luyen_mo_hinh()

## 2.8 Trực Quan Hóa & Phân Tích Đường Cong Mất Mát (`Loss Curve Visualization`)
Đoạn mã này thực hiện việc đọc tệp lịch sử huấn luyện đã lưu (`training_history.json`) và tự động trực quan hóa tiến trình hội tụ của mô hình U-Net:
* **Tải dữ liệu lịch sử**: Sử dụng thư viện `json` để khôi phục từ điển `history` chứa các giá trị mất mát qua từng kỷ nguyên.
* **Vẽ biểu đồ so sánh trực quan (`Matplotlib`)**:
  * Trực quan hóa đồng thời hai đường cong: **Train Loss** (xu hướng học tập trên tập huấn luyện) và **Val Loss** (khả năng khái quát hóa trên tập kiểm định).
  * Tích hợp các điểm đánh dấu (`marker='o'`) và lưới tọa độ (`grid=True`) giúp dễ dàng nhận biết điểm tối ưu hoặc hiện tượng học vẹt (overfitting) nếu có.
* **Lưu trữ tài nguyên báo cáo**: Tự động lưu biểu đồ chất lượng cao với độ phân giải `dpi=300` dưới tên `loss_curve.png`, sẵn sàng chèn trực tiếp vào báo cáo học thuật hoặc tiểu luận đồ án.

In [ ]:
import json
import matplotlib.pyplot as plt

# Đọc file lịch sử
with open('training_history.json', 'r', encoding='utf-8') as f:
    history = json.load(f)

# Vẽ biểu đồ Loss
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'], label='Val Loss', marker='o')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve.png', dpi=300) # Lưu ảnh ra file để đưa vào báo cáo
plt.show()

## 2.9 Trực Quan Hóa Toàn Diện Tiến Trình Huấn Luyện (`Comprehensive Training Curves`)
Đoạn mã này nâng cấp quy trình trực quan hóa bằng cách gom nhóm toàn bộ các chỉ số quan trọng vào một khung hình gồm hai biểu đồ song song (`1 hàng x 2 cột`), cung cấp cái nhìn tổng thể và trực quan nhất về hiệu năng của mô hình U-Net:
* **Xác định trục thời gian linh hoạt**: Tự động tạo danh sách `epochs` dựa trên tổng số kỷ nguyên thực tế ghi nhận được từ tệp lịch sử (`training_history.json`).
* **Biểu đồ 1 (Loss Curves - `ax1`)**: So sánh trực diện tiến trình giảm của **Train Loss** và **Val Loss** qua từng kỷ nguyên, giúp đánh giá nhanh chóng tốc độ hội tụ và kiểm soát hiện tượng quá khớp (overfitting).
* **Biểu đồ 2 (Evaluation Metrics - `ax2`)**: Theo dõi đồng thời hệ số tương đồng **Val Dice** và độ đo **Val IoU** trên tập kiểm định, với trục tung được cố định giới hạn từ $0$ đến $1.05$ (`plt.set_ylim`) để phản ánh chính xác chất lượng phân đoạn vùng tổn thương u não.
* **Tối ưu hóa và xuất bản tài nguyên**: Tự động căn chỉnh khoảng cách hợp lý (`plt.tight_layout()`) và xuất tệp ảnh chất lượng cao dưới tên `training_curves.png`, sẵn sàng đưa vào các chương mục đánh giá kết quả của đồ án.

In [ ]:
import json
import matplotlib.pyplot as plt

# 1. Đọc lại file lịch sử huấn luyện
with open('training_history.json', 'r', encoding='utf-8') as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss']) + 1)

# 2. Thiết lập khung vẽ (1 hàng, 2 cột biểu đồ)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Biểu đồ 1: Loss (Train Loss vs Val Loss) ---
ax1.plot(epochs, history['train_loss'], label='Train Loss', marker='o', color='blue', linewidth=2)
ax1.plot(epochs, history['val_loss'], label='Val Loss', marker='s', color='orange', linewidth=2)
ax1.set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Loss', fontsize=10)
ax1.legend(fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.6)

# --- Biểu đồ 2: Metrics (Val Dice & Val IoU) ---
ax2.plot(epochs, history['val_dice'], label='Val Dice', marker='^', color='green', linewidth=2)
ax2.plot(epochs, history['val_iou'], label='Val IoU', marker='d', color='purple', linewidth=2)
ax2.set_title('Validation Dice Score & IoU', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=10)
ax2.set_ylabel('Score', fontsize=10)
ax2.legend(fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.set_ylim(0, 1.05)  # Giới hạn trục y từ 0 đến 1 cho Dice/IoU

# 3. Tối ưu khoảng cách và lưu file ảnh với dpi=200
plt.tight_layout()
plt.savefig('training_curves.png', dpi=200)
print("Đã vẽ và lưu biểu đồ thành công vào file 'training_curves.png'!")

# Hiển thị biểu đồ ngay trong notebook
plt.show()

## 2.10 Đo Lường Thời Gian Huấn Luyện & Ghi Nhận Thông Tin Phần Cứng (`Training Time & Hardware Logging`)
Đoạn mã này tự động hóa công đoạn thống kê siêu dữ liệu (metadata) của quá trình huấn luyện, cung cấp thông số minh bạch về tài nguyên tính toán cho báo cáo nghiên cứu:
* **Đo lường và định dạng thời gian thực thi**: Sử dụng module `time` để tính tổng khoảng thời gian trôi qua từ lúc bắt đầu đến khi kết thúc quá trình học tập, sau đó quy đổi linh hoạt sang định dạng dễ đọc (`giờ, phút, giây`).
* **Truy xuất thông tin cấu hình phần cứng tự động**: Kiểm tra trạng thái thiết bị thông qua `torch.cuda`, tự động trích xuất tên card đồ họa (`torch.cuda.get_device_name`) và dung lượng bộ nhớ VRAM (`total_memory`) để ghi nhận chính xác phần cứng thực tế sử dụng trên Kaggle (ví dụ: T4, P100, hoặc T4 x2).
* **Cập nhật siêu dữ liệu vào tệp lịch sử chung**: Bổ sung hai trường thông tin `training_time` và `hardware` trực tiếp vào từ điển `history`, sau đó ghi đè lên tệp `training_history.json`, giúp đồng bộ toàn bộ kết quả thực nghiệm và sẵn sàng đưa vào bảng tổng kết thông số phần cứng trong chương đánh giá của đồ án.

In [ ]:
import time
import torch

# ... (đoạn code train của bạn ở trên) ...

    # Giả sử bạn bắt đầu đếm giờ từ đầu hàm huan_luyen_mo_hinh() hoặc trước vòng lặp
    # (Bạn có thể đặt start_time = time.time() ngay đầu hàm)

    # Sau khi kết thúc huấn luyện:
    end_time = time.time()
    elapsed_time = end_time - start_time  # tính bằng giây
    
    # Quy đổi ra giờ, phút, giây
    hours = int(elapsed_time // 3600)
    minutes = int((elapsed_time % 3600) // 60)
    seconds = int(elapsed_time % 60)
    time_str = f"{hours}h {minutes}m {seconds}s" if hours > 0 else f"{minutes}m {seconds}s"
    
    print(f"\n⏱️ Tổng thời gian huấn luyện: {time_str}")

    # Lấy thông tin phần cứng GPU
    hardware_info = "CPU"
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3) # tính bằng GB
        hardware_info = f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)"
    print(f"Cấu hình thiết bị: {hardware_info}")

    # Cập nhật thông tin vào file history để lưu chung luôn
    history['training_time'] = time_str
    history['hardware'] = hardware_info

    with open('training_history.json', 'w', encoding='utf-8') as f:
        json.dump(history, f, indent=4)
    print("Đã cập nhật thời gian và phần cứng vào training_history.json!")

## 2.11 Xây Dựng Pipeline Otsu Kết Hợp Hậu Xử Lý Hình Thái Học (`xu_ly_otsu_va_hau_xu_ly`)
Hàm này hoàn thiện phương pháp cơ sở (baseline) truyền thống bằng cách tích hợp các kỹ thuật xử lý ảnh nâng cao, giúp cải thiện đáng kể chất lượng phân đoạn so với thuật toán Otsu nguyên bản:
* **Tiền xử lý và làm mờ thích ứng**: Chuyển đổi định dạng an toàn về `uint8` và áp dụng bộ lọc Gauss (`cv2.GaussianBlur`) để làm mượt các biến động cường độ cục bộ trước khi tính toán ngưỡng.
* **Phân ngưỡng tự động Otsu (`cv2.THRESH_OTSU`)**: Xác định ngưỡng cường độ tối ưu dựa trên phân bố lược đồ xám (histogram) của ảnh MRI.
* **Hậu xử lý hình thái học (`Morphological Operations`)**:
  * **Phép mở (`Morphology Opening`)**: Sử dụng cấu trúc hình elip để loại bỏ các đốm nhiễu sáng rải rác bên ngoài vùng tổn thương.
  * **Phép đóng (`Morphology Closing`)**: Lấp đầy các lỗ rỗng bên trong khối u, đảm bảo vùng phân đoạn liền mạch và đồng nhất.
* **Lọc thành phần liên thông lớn nhất (`Connected Components Analysis`)**: 
  * Quét và phân loại các thành phần liên thông (`cv2.connectedComponentsWithStats`), tự động loại bỏ các nhiễu nền nhỏ dựa trên diện tích (`cv2.CC_STAT_AREA`).
  * Chỉ giữ lại vùng có diện tích lớn nhất làm mặt nạ dự đoán chính thức cho khối u (`tumor_mask`).
* **Chuẩn hóa đầu ra**: Chuyển đổi kết quả về dạng ma trận số thực `float32` trong khoảng $[0.0, 1.0]$, giúp tương thích hoàn hảo với định dạng chuẩn của các mô hình học sâu U-Net khi tính toán các độ đánh giá (Dice, IoU).

In [ ]:
import cv2
import numpy as np

def xu_ly_otsu_va_hau_xu_ly(img):
    """
    Pipeline Otsu truyền thống + Hậu xử lý:
    1. Chuẩn hóa ảnh về uint8 [0, 255].
    2. Phân đoạn ngưỡng Otsu.
    3. Hậu xử lý hình thái học (Morphology: Opening để xóa nhiễu, Closing để lấp lỗ thủng).
    4. Lọc và giữ lại thành phần liên thông lớn nhất (loại bỏ các đốm nhiễu nhỏ xung quanh).
    """
    # 1. Đảm bảo ảnh ở định dạng uint8 [0, 255]
    if img.dtype != np.uint8:
        img_uint8 = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    else:
        img_uint8 = img

    # (Tùy chọn) Khử nhiễu nhẹ trước khi tìm ngưỡng Otsu
    blurred = cv2.GaussianBlur(img_uint8, (5, 5), 0)

    # 2. Phân ngưỡng Otsu (tự động tìm ngưỡng tối ưu dựa trên histogram)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. Hậu xử lý hình thái học (Morphological Operations)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    # Opening: Xóa các đốm nhiễu sáng nhỏ bên ngoài
    opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    # Closing: Lấp đầy các lỗ đen nhỏ bên trong vùng khối u
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=1)

    # 4. Giữ lại thành phần liên thông lớn nhất (Connected Components)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(closed, connectivity=8)
    
    tumor_mask = np.zeros_like(closed)
    if num_labels > 1:
        # Bỏ qua background (label 0), tìm label có diện tích (area) lớn nhất
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        # Chỉ giữ lại vùng lớn nhất đó
        tumor_mask[labels == largest_label] = 255

    # Trả về dạng float32 trong khoảng [0.0, 1.0] để khớp chuẩn với U-Net
    return (tumor_mask > 127).astype(np.float32)

# --- Kiểm tra thử trên một ảnh mẫu từ tập val ---
# anh_mau, mask_that = tap_val[0]
# mask_du_doan_otsu = xu_ly_otsu_va_hau_xu_ly(anh_mau.squeeze().numpy())

# 3.Evaluation, Post-processing & Visualization (Đánh giá & Xuất kết quả)

## 3.1 Xây Dựng Hàm Đánh Giá Hiệu Năng Chuyên Sâu (`tinh_dice_iou`)
Hàm này cung cấp công cụ định lượng chuẩn xác để đánh giá chất lượng phân đoạn trên tập kiểm tra (`Test Set`), chuyển đổi các mặt nạ nhị phân thành các chỉ số toán học khách quan:
* **Làm phẳng không gian dữ liệu (`Flatten`)**: Sử dụng phương thức `.flatten()` để chuyển đổi các ma trận mặt nạ không gian hai chiều (`H x W`) thành các véc-tơ một chiều, chuẩn bị cho việc tính toán tổng hợp pixel toàn cục.
* **Tính toán phần giao nhau (`Intersection`)**: Thực hiện phép nhân phần tử và tính tổng (`.sum()`) để xác định số lượng điểm ảnh mà mô hình dự đoán trùng khớp hoàn toàn với nhãn thực tế từ chuyên gia.
* **Hệ số tương đồng Dice Coefficient**: Tính toán mức độ chồng lấp giữa hai vùng dựa trên công thức chuẩn có chứa hằng số làm mượt (`smooth=1e-5`) nhằm tránh lỗi chia cho số không khi ảnh hoàn toàn là nền đen.
* **Độ đo Intersection over Union (IoU / Jaccard Index)**: Đo lường tỷ lệ phần giao nhau trên phần hợp (`union = du_doan.sum() + thuc_te.sum() - giao_nhau`), cung cấp một tiêu chuẩn khắt khe và phổ biến trong các bài toán phân đoạn ảnh y tế.

In [ ]:
def tinh_dice_iou(du_doan, thuc_te, smooth=1e-5):
    """
    Tính Dice Score và IoU trên tập dữ liệu Test.
    du_doan và thuc_te là mảng numpy nhị phân (0 hoặc 1).
    """
    du_doan = du_doan.flatten()
    thuc_te = thuc_te.flatten()
    
    giao_nhau = (du_doan * thuc_te).sum()
    
    # Dice Coefficient
    dice = (2. * giao_nhau + smooth) / (du_doan.sum() + thuc_te.sum() + smooth)
    
    # Intersection over Union (IoU)
    union = du_doan.sum() + thuc_te.sum() - giao_nhau
    iou = (giao_nhau + smooth) / (union + smooth)
    
    return dice, iou


## 3.2 Xây Dựng Hàm Hậu Xử Lý Mặt Nạ Dự Đoán (`hau_xu_ly_mask`)
Hàm này cung cấp bước tinh chỉnh cuối cùng nhằm làm sạch các mặt nạ nhị phân do mô hình U-Net dự đoán, loại bỏ nhiễu và đảm bảo tính liền mạch của vùng ranh giới khối u trước khi đánh giá định lượng:
* **Chuyển đổi không gian định dạng**: Quy đổi ma trận dự đoán từ miền số thực $[0.0, 1.0]$ sang thang xám `uint8` $[0, 255]$ để tương thích hoàn toàn với các hàm xử lý hình thái học của OpenCV.
* **Cấu trúc nhân hình học (`Kernel 3x3`)**: Thiết lập nhân hình chữ nhật kích thước $3 \times 3$ làm phần tử cấu trúc cơ bản cho các phép biến đổi hình thái học.
* **Kỹ thuật hình thái học kết hợp (`Morphological Operations`)**:
  * **Phép mở (`Morphology Opening`)**: Triệt tiêu hiệu quả các đốm nhiễu sáng nhỏ rời rạc xuất hiện ngẫu nhiên ở vùng nền xung quanh.
  * **Phép đóng (`Morphology Closing`)**: Lấp đầy các khoảng trống hoặc lỗ thủng nhỏ lọt thỏm bên trong khối u, giúp vùng tổn thương trở nên đồng nhất và hoàn chỉnh hơn.
* **Chuẩn hóa đầu ra định dạng chuẩn**: Khôi phục mặt nạ về dạng ma trận nhị phân chuẩn với giá trị nhị phân `0.0` và `1.0` (kiểu dữ liệu `float32`), sẵn sàng cho quá trình tính toán các chỉ số đánh giá hiệu năng (Dice, IoU) trên tập kiểm tra (`Test Set`).

In [ ]:
# ==========================================
# 2. HÀM HẬU XỬ LÝ (POST-PROCESSING)
# ==========================================
def hau_xu_ly_mask(mask_du_doan):
    """
    Dùng các phép toán hình học (Morphological Operations) để xóa đốm nhiễu thừa.
    - Phép mở (Opening): Loại bỏ các đốm nhiễu nhỏ rời rạc bên ngoài.
    - Phép đóng (Closing): Lấp đầy các khoảng trống nhỏ bên trong khối u.
    """
    # Đổi về kiểu ảnh uint8 với dải 0-255 để xử lý với OpenCV
    mask_bin = (mask_du_doan * 255).astype(np.uint8)
    
    # Tạo kernel hình học kích thước 3x3
    kernel = np.ones((3, 3), np.uint8)
    
    # Áp dụng phép Opening và Closing
    mask_sach = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, kernel)
    mask_sach = cv2.morphologyEx(mask_sach, cv2.MORPH_CLOSE, kernel)
    
    # Trả về dạng mảng nhị phân 0 và 1 (kiểu float32 để đồng bộ)
    return (mask_sach > 0).astype(np.float32)

## 3.3 Xây Dựng Hàm Trực Quan Hóa & Đánh Giá Định Lượng Toàn Diện Trên Tập Test (`chay_visualize_va_danh_gia`)
Hàm này tích hợp toàn bộ quy trình suy luận (inference), đánh giá số liệu và trực quan hóa kết quả phân đoạn trên tập kiểm tra (`Test Set`), tạo ra góc nhìn trực quan và khách quan nhất về năng lực khái quát hóa của mô hình U-Net:
* **Nạp Trọng Số & Tách Tập Kiểm Tra**: 
  * Tự động kiểm tra và nạp tệp trọng số tốt nhất đã huấn luyện (`best_model.pth`), đảm bảo mô hình chuyển sang trạng thái đánh giá (`model.eval()`).
  * Tái tạo chính xác tập kiểm tra thông qua thuật toán phân chia cấp độ bệnh nhân, đảm bảo không có hiện tượng rò rỉ dữ liệu (`data leakage`).
* **Suy Luận Học Sâu & Hậu Xử Lý**: 
  * Chuyển đổi mẫu ảnh thử nghiệm sang định dạng tensor chuẩn, thực hiện lan truyền tiến không lưu gradient (`torch.no_grad()`).
  * Áp dụng hàm kích hoạt `sigmoid` và ngưỡng quyết định `0.5`, tiếp tục chạy qua hàm hậu xử lý hình thái học (`hau_xu_ly_mask`) để loại bỏ nhiễu và làm sạch mặt nạ dự đoán.
* **Đánh Giá Định Lượng Chuyên Sâu**: Tính toán chính xác các hệ số đánh giá y khoa tiêu chuẩn (**Dice Score** và **IoU Score**) giữa mặt nạ dự đoán và mặt nạ thực tế, in kết quả chi tiết ra màn hình console.
* **Bảng Trực Quan Hóa 4 Cột Song Song (`4-Column Comparative Layout`)**:
  * **Cột 1 (`Ảnh MRI Đầu Vào`)**: Lát cắt MRI thô sau khi đã qua bước tiền xử lý chuẩn hóa.
  * **Cột 2 (`Mask Thực Tế`)**: Nhãn chuẩn vàng (`Ground Truth`) do các chuyên gia y tế cung cấp.
  * **Cột 3 (`U-Net Hậu Xử Lý`)**: Vùng tổn thương do mô hình học sâu phân đoạn kèm theo chỉ số Dice chi tiết.
  * **Cột 4 (`Overlay So Sánh`)**: Chồng lớp trực tiếp trên ảnh gốc với đường viền xanh dương đại diện cho Ground Truth và đường viền đỏ đại diện cho dự đoán của U-Net, giúp đánh giá trực diện mức độ chồng lấp và sai lệch ranh giới khối u.

In [ ]:
# ==========================================
# 3. SCRIPT XUẤT ẢNH KẾT QUẢ (VISUALIZATION 4 CỘT)
# ==========================================
def chay_visualize_va_danh_gia(so_file_test_idx=0):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Tải mô hình U-Net và nạp trọng số tốt nhất đã huấn luyện (.pth)
    model = UNet(in_channels=1, out_channels=1).to(device)
    duong_dan_weight = 'best_model.pth'
    
    if not os.path.exists(duong_dan_weight):
        print(f"Cảnh báo: Không tìm thấy file trọng số '{duong_dan_weight}'. Hãy chạy huấn luyện trước khi gọi hàm dự đoán thực tế!")
        return

    model.load_state_dict(torch.load(duong_dan_weight, map_location=device))
    model.eval()

    # Lấy danh sách file test bằng cách chia tập đúng theo thuật toán cố định
    danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
    nhom = gom_theo_benh_nhan(danh_sach_file)
    _, _, benh_nhan_test = chia_benh_nhan(nhom)
    file_test = lay_file_cua_benh_nhan(benh_nhan_test, nhom)

    if len(file_test) == 0:
        print("Không tìm thấy file nào trong tập Test!")
        return

    # Chọn 1 file mẫu từ tập Test để kiểm tra
    duong_dan_file_test = file_test[so_file_test_idx]
    
    # Đọc và tiền xử lý ảnh
    anh_goc, mask_goc = doc_file_mat(duong_dan_file_test)
    anh_xu_ly, mask_xu_ly = tien_xu_ly(anh_goc, mask_goc)

    # Đưa ảnh về định dạng Tensor chuẩn để đưa vào mạng U-Net
    anh_tensor = anh_xu_ly.astype(np.float32) / 255.0
    anh_tensor = anh_tensor.reshape(1, 1, 256, 256)
    anh_tensor = torch.from_numpy(anh_tensor).to(device)

    # Dự đoán bằng mô hình
    with torch.no_grad():
        dau_ra = model(anh_tensor)
        du_doan_prob = torch.sigmoid(dau_ra)
        du_doan_mask = (du_doan_prob > 0.5).float()
        
    du_doan_np = du_doan_mask.squeeze().cpu().numpy()
    
    # Thực hiện hậu xử lý xóa nhiễu
    du_doan_sach = hau_xu_ly_mask(du_doan_np)

    # Tính toán các chỉ số đánh giá y khoa
    dice, iou = tinh_dice_iou(du_doan_sach, mask_xu_ly)
    print(f"--- KẾT QUẢ ĐÁNH GIÁ TRÊN ẢNH TEST ---")
    print(f"Dice Score : {dice:.4f}")
    print(f"IoU Score  : {iou:.4f}")

    # Vẽ bảng kết quả gồm 4 cột trực quan
    plt.figure(figsize=(16, 4))

    # Cột 1: Ảnh MRI đầu vào sau tiền xử lý
    plt.subplot(1, 4, 1)
    plt.imshow(anh_xu_ly, cmap='gray')
    plt.title('1. Ảnh MRI Đầu Vào')
    plt.axis('off')

    # Cột 2: Mask thực tế (Ground Truth từ bác sĩ/dataset)
    plt.subplot(1, 4, 2)
    plt.imshow(mask_xu_ly, cmap='gray')
    plt.title('2. Mask Thực Tế')
    plt.axis('off')

    # Cột 3: Kết quả dự đoán của U-Net sau khi hậu xử lý
    plt.subplot(1, 4, 3)
    plt.imshow(du_doan_sach, cmap='gray')
    plt.title(f'3. U-Net (Hậu xử lý)\n(Dice: {dice:.2f})')
    plt.axis('off')

    # Cột 4: Ảnh chồng khép (Overlay viền xanh: Thực tế | Đỏ: Dự đoán)
    plt.subplot(1, 4, 4)
    plt.imshow(anh_xu_ly, cmap='gray')
    plt.contour(mask_xu_ly, colors='blue', linewidths=1)
    plt.contour(du_doan_sach, colors='red', linewidths=1)
    plt.title('4. Overlay\n(Xanh: GT | Đỏ: Pred)')
    plt.axis('off')

    plt.tight_layout()
    plt.show()


## 3.4 Khối Thực Thi Thử Nghiệm Trực Quan Hóa Kết Quả Trên Tập Kiểm Tra (`Test Inference Execution`)
Đoạn mã này thực thi việc gọi hàm đánh giá và trực quan hóa toàn diện trên mẫu dữ liệu đầu tiên (`so_file_test_idx=0`) thuộc tập kiểm tra (`Test Set`):
* **Kích hoạt quy trình khép kín**: Tự động nạp mô hình U-Net đã huấn luyện tối ưu, thực hiện suy luận trên ảnh MRI thực tế chưa từng xuất hiện trong quá trình huấn luyện.
* **Xuất bản báo cáo trực quan 4 cột**: Hiển thị trực tiếp bảng so sánh trực quan gồm ảnh gốc, mặt nạ chuẩn y tế, mặt nạ dự đoán của mô hình và hình ảnh chồng lớp (overlay contour), kèm theo các chỉ số định lượng khắt khe như **Dice Score** và **IoU Score**, hoàn thiện trọn vẹn quy trình nghiên cứu và đánh giá mô hình phân đoạn khối u não.

In [ ]:

if __name__ == '__main__':
    chay_visualize_va_danh_gia(so_file_test_idx=0)

## 3.5 Xây Dựng Pipeline Đánh Giá Định Lượng Toàn Diện Trên Tập Test (`danh_gia_toan_bo_test`)
Đoạn mã này hiện thực hóa quy trình kiểm định so sánh hiệu năng quy mô lớn trên toàn bộ tập kiểm tra (`Test Set`), thiết lập cơ sở khoa học vững chắc cho việc đối chiếu giữa phương pháp truyền thống và mô hình học sâu đề xuất:
* **Nạp mô hình U-Net tối ưu**: Tự động tải trọng số tốt nhất đã huấn luyện (`best_dice_model.pth`) lên thiết bị phần cứng phù hợp (`GPU CUDA` hoặc `CPU`) và chuyển sang chế độ đánh giá (`model.eval()`).
* **Thiết lập khung dữ liệu thử nghiệm chuẩn hóa**: Trích xuất chính xác tập kiểm tra thông qua thuật toán phân chia cấp độ bệnh nhân, đảm bảo tính khách quan và độc lập tuyệt đối giữa tập huấn luyện và kiểm định.
* **Đánh giá đa phương pháp song song (`Multi-Method Evaluation Pipeline`)**: Duyệt qua từng lát cắt MRI trong tập test để tính toán đồng thời các chỉ số trên 4 cấu hình:
  1. **Otsu Thuần**: Phương pháp phân ngưỡng tự động truyền thống.
  2. **Otsu + Hậu Xử Lý**: Kết hợp bộ lọc Gaussian, hình thái học (`Opening`/`Closing`) và lọc thành phần liên thông lớn nhất.
  3. **U-Net (Thô)**: Mô hình học sâu U-Net suy luận trực tiếp qua hàm kích hoạt ngưỡng xác suất `0.5`.
  4. **U-Net + Hậu Xử Lý**: Mô hình U-Net kết hợp tinh chỉnh hình thái học (giải pháp đề xuất tối ưu).
* **Thống kê y khoa chuyên sâu (`Mean ± Standard Deviation`)**: 
  * Sử dụng các độ đo tiêu chuẩn: **Dice Coefficient**, **Intersection over Union (IoU)**, **Precision**, và **Recall** để định lượng mức độ trùng khớp vùng tổn thương.
  * Tự động tính toán giá trị trung bình (`mean`) và độ lệch chuẩn (`std`) cho từng phương pháp trên toàn bộ tập test, in ra màn hình console dưới dạng bảng báo cáo khoa học chuẩn mực, sẵn sàng đưa vào chương đánh giá kết quả của đồ án.

In [ ]:
import sys
import os
import torch
import cv2
import numpy as np
from unet import UNet


# ==========================================
# 1. HÀM TÍNH TOÁN CHỈ SỐ ĐÁNH GIÁ (DICE & IoU)
# ==========================================
def tinh_cac_chi_so(du_doan, thuc_te, smooth=1e-5):
    """Tính Dice Coefficient, IoU, Precision và Recall chuẩn y khoa."""
    du_doan = du_doan.flatten()
    thuc_te = thuc_te.flatten()

    tp = (du_doan * thuc_te).sum()
    fp = (du_doan * (1 - thuc_te)).sum()
    fn = ((1 - du_doan) * thuc_te).sum()
    giao_nhau = tp  # Giao nhau (Intersection) chính là True Positive
    
    # Dice Score
    dice = (2. * giao_nhau + smooth) / (du_doan.sum() + thuc_te.sum() + smooth)
    
    # IoU Score
    union = du_doan.sum() + thuc_te.sum() - giao_nhau
    iou = (giao_nhau + smooth) / (union + smooth)

    # Precision
    precision = (tp + smooth) / (tp + fp + smooth)

    # Recall
    recall = (tp + smooth) / (tp + fn + smooth)
    
    return dice, iou, precision, recall


# ==========================================
# 2. CÁC PHƯƠNG PHÁP PHÂN ĐOẠN (BASELINE & ĐỀ XUẤT)
# ==========================================
def chay_otsu_thuan(img_uint8):
    """Phương pháp 1: Otsu thuần (truyền thống)"""
    _, thresh = cv2.threshold(img_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return (thresh > 127).astype(np.float32)


def chay_otsu_va_hau_xu_ly(img_uint8):
    """Phương pháp 2: Otsu kết hợp Hậu xử lý (Morphology + Lọc vùng lớn nhất)"""
    blurred = cv2.GaussianBlur(img_uint8, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Hậu xử lý hình thái học
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=1)
    
    # Giữ lại thành phần liên thông lớn nhất
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    tumor_mask = np.zeros_like(closed)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        tumor_mask[labels == largest_label] = 255
        
    return (tumor_mask > 127).astype(np.float32)


def chay_unet_hau_xu_ly(mask_du_doan):
    """Hậu xử lý cho U-Net: Xóa đốm nhiễu nhỏ xung quanh"""
    mask_bin = (mask_du_doan * 255).astype(np.uint8)
    kernel = np.ones((3, 3), np.uint8)
    opened = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN, kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return (closed > 0).astype(np.float32)


# ==========================================
# 3. HÀM CHÍNH ĐÁNH GIÁ TRÊN TOÀN BỘ TẬP TEST
# ==========================================
def danh_gia_toan_bo_test():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Đang sử dụng thiết bị: {device}")

    # Load mô hình U-Net
    model = UNet(in_channels=1, out_channels=1).to(device)
    duong_dan_weight = 'best_dice_model.pth' # Hoặc 'best_model.pth' tùy bạn đặt
    
    if not os.path.exists(duong_dan_weight):
        print(f"Lỗi: Không tìm thấy file trọng số '{duong_dan_weight}'!")
        return

    model.load_state_dict(torch.load(duong_dan_weight, map_location=device))
    model.eval()

    # Lấy danh sách file test theo đúng phân chia chuẩn của đồ án
    danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
    nhom = gom_theo_benh_nhan(danh_sach_file)
    _, _, benh_nhan_test = chia_benh_nhan(nhom)
    file_test = lay_file_cua_benh_nhan(benh_nhan_test, nhom)

    if len(file_test) == 0:
        print("Không tìm thấy file nào trong tập Test!")
        return
        
    print(f"Đang tiến hành đánh giá trên toàn bộ tập Test ({len(file_test)} ảnh)...")
    
    # Định nghĩa từ điển loại u (nếu cần dùng mở rộng)
    TEN_LOAI_U = {
        1: "Meningioma",
        2: "Glioma",
        3: "Pituitary"
    }

    # Danh sách lưu kết quả cho từng phương pháp (Đồng bộ chính tả key)
    metrics = {
        "Otsu Thuần": {"dice": [], "iou": [], "precision": [], "recall": []},
        "Otsu + Hậu xử lý": {"dice": [], "iou": [], "precision": [], "recall": []},
        "U-Net (Thô)": {"dice": [], "iou": [], "precision": [], "recall": []},
        "U-Net + Hậu xử lý": {"dice": [], "iou": [], "precision": [], "recall": []},
    }

    # Duyệt qua toàn bộ tập Test
    with torch.no_grad():
        for duong_dan in file_test:
            # 1. Đọc và tiền xử lý
            anh_goc, mask_goc = doc_file_mat(duong_dan)
            anh_xu_ly, mask_xu_ly = tien_xu_ly(anh_goc, mask_goc)

            # Chuẩn bị ảnh uint8 cho phương pháp Otsu truyền thống
            anh_uint8 = (anh_xu_ly * 255).astype(np.uint8) if anh_xu_ly.max() <= 1.0 else anh_xu_ly.astype(np.uint8)

            # --- Phương pháp 1: Otsu thuần ---
            pred_otsu1 = chay_otsu_thuan(anh_uint8)
            d1, i1, pr1, r1 = tinh_cac_chi_so(pred_otsu1, mask_xu_ly)
            metrics["Otsu Thuần"]["dice"].append(d1)
            metrics["Otsu Thuần"]["iou"].append(i1)
            metrics["Otsu Thuần"]["precision"].append(pr1)
            metrics["Otsu Thuần"]["recall"].append(r1)

            # --- Phương pháp 2: Otsu + Hậu xử lý ---
            pred_otsu2 = chay_otsu_va_hau_xu_ly(anh_uint8)
            d2, i2, pr2, r2 = tinh_cac_chi_so(pred_otsu2, mask_xu_ly)
            metrics["Otsu + Hậu xử lý"]["dice"].append(d2)
            metrics["Otsu + Hậu xử lý"]["iou"].append(i2)
            metrics["Otsu + Hậu xử lý"]["precision"].append(pr2)
            metrics["Otsu + Hậu xử lý"]["recall"].append(r2)

            # --- Chuẩn bị dữ liệu cho U-Net ---
            anh_tensor = anh_xu_ly.astype(np.float32)
            if anh_tensor.max() > 1.0:
                anh_tensor = anh_tensor / 255.0
            anh_tensor = anh_tensor.reshape(1, 1, 256, 256)
            anh_tensor = torch.from_numpy(anh_tensor).to(device)

            # Dự đoán từ U-Net
            dau_ra = model(anh_tensor)
            pred_prob = torch.sigmoid(dau_ra)
            pred_unet_raw = (pred_prob > 0.5).float().squeeze().cpu().numpy()

            # --- Phương pháp 3: U-Net thô ---
            d3, i3, pr3, r3 = tinh_cac_chi_so(pred_unet_raw, mask_xu_ly)
            metrics["U-Net (Thô)"]["dice"].append(d3)
            metrics["U-Net (Thô)"]["iou"].append(i3)
            metrics["U-Net (Thô)"]["precision"].append(pr3)
            metrics["U-Net (Thô)"]["recall"].append(r3)

            # --- Phương pháp 4: U-Net + Hậu xử lý ---
            pred_unet_post = chay_unet_hau_xu_ly(pred_unet_raw)
            d4, i4, pr4, r4 = tinh_cac_chi_so(pred_unet_post, mask_xu_ly)
            metrics["U-Net + Hậu xử lý"]["dice"].append(d4)
            metrics["U-Net + Hậu xử lý"]["iou"].append(i4)
            metrics["U-Net + Hậu xử lý"]["precision"].append(pr4)
            metrics["U-Net + Hậu xử lý"]["recall"].append(r4)

    # ==========================================
    # 4. XUẤT BẢNG TỔNG KẾT KẾT QUẢ KHOA HỌC
    # ==========================================
    print("\n" + "="*115)
    print("BẢNG TỔNG KẾT SO SÁNH HIỆU SUẤT TRÊN TOÀN BỘ TẬP TEST (Mean ± Std)")
    print("="*115)
    print(f"{'Phương pháp':<22} | {'Dice (Mean ± Std)':<22} | {'IoU (Mean ± Std)':<22} | {'Precision':<18} | {'Recall':<18}")
    print("-" * 115)

    for ten_pp, data in metrics.items():
        d_mean, d_std = np.mean(data["dice"]), np.std(data["dice"])
        i_mean, i_std = np.mean(data["iou"]), np.std(data["iou"])
        pr_mean, pr_std = np.mean(data["precision"]), np.std(data["precision"])
        r_mean, r_std = np.mean(data["recall"]), np.std(data["recall"])
        
        print(f"{ten_pp:<22} | {d_mean:.4f} ± {d_std:.4f}      | {i_mean:.4f} ± {i_std:.4f}      | {pr_mean:.4f} ± {pr_std:.4f} | {r_mean:.4f} ± {r_std:.4f}")

    print("="*115)
    print("Đã hoàn thành đánh giá toàn bộ tập Test thành công!")


if __name__ == '__main__':
    danh_gia_toan_bo_test()

In [ ]:
import pandas as pd
import numpy as np

# Giả sử trong cell chạy evaluation trước đó, biến `metrics` đã được lưu lại
# Nếu bạn chạy gộp chung, đoạn này sẽ lấy dữ liệu từ `metrics` để xuất DataFrame.

# Chuẩn bị danh sách chứa dữ liệu để tạo DataFrame
data_export = []

for ten_pp, data in metrics.items():
    d_mean, d_std = np.mean(data["dice"]), np.std(data["dice"])
    i_mean, i_std = np.mean(data["iou"]), np.std(data["iou"])
    pr_mean, pr_std = np.mean(data["precision"]), np.std(data["precision"])
    r_mean, r_std = np.mean(data["recall"]), np.std(data["recall"])
    
    # Định dạng theo chuẩn Mean ± Std
    data_export.append({
        "Phương pháp": ten_pp,
        "Dice Score (Mean ± Std)": f"{d_mean:.4f} ± {d_std:.4f}",
        "IoU Score (Mean ± Std)": f"{i_mean:.4f} ± {i_std:.4f}",
        "Precision (Mean ± Std)": f"{pr_mean:.4f} ± {pr_std:.4f}",
        "Recall (Mean ± Std)": f"{r_mean:.4f} ± {r_std:.4f}"
    })

# Tạo bảng Pandas DataFrame
df_ket_qua = pd.DataFrame(data_export)

# 1. In bảng ra màn hình để copy nhanh
print("\n" + "="*95)
print("BẢNG SỐ LIỆU THỐNG KÊ PHỤC VỤ BÁO CÁO ĐỒ ÁN")
print("="*95)
print(df_ket_qua.to_string(index=False))
print("="*95)

# 2. Xuất tự động thành file CSV lưu trên Kaggle
file_csv_path = "ket_qua_danh_gia.csv"
df_ket_qua.to_csv(file_csv_path, index=False, encoding='utf-8-sig')
print(f"Đã lưu thành công file CSV tại: '{file_csv_path}'")
print("Bạn có thể nhìn sang thanh bên phải (Data/Output) của Kaggle để tải file này về máy!")

## 3.6 Xây Dựng Pipeline Trực Quan Hóa Ca Tiêu Biểu (`Best, Median, Worst Case Analysis`)
Đoạn mã này thực hiện việc quét toàn bộ tập kiểm tra (`Test Set`), đánh giá định lượng bằng hệ số Dice Coefficient cho từng mẫu, từ đó tự động trích xuất và hiển thị ba trường hợp đại diện: **Tốt nhất (Best)**, **Trung vị (Median)**, và **Xấu nhất (Worst)**:
* **Quét và xếp hạng toàn bộ tập dữ liệu**: Duyệt qua từng lát cắt MRI trong tập test, thu thập kết quả dự đoán từ cả hai phương pháp (Otsu cải tiến và U-Net đề xuất) rồi tính điểm Dice của U-Net làm tiêu chuẩn sắp xếp.
* **Trích xuất các ca đặc trưng**: Sắp xếp danh sách kết quả theo thứ tự tăng dần của Dice Score để chọn ra:
  * **Ca Tốt nhất (`Best Case`)**: Mẫu có độ trùng khớp cao nhất, minh chứng cho năng lực phân đoạn tối ưu của U-Net.
  * **Ca Trung vị (`Median Case`)**: Mẫu phản ánh hiệu năng trung bình, đại diện cho độ chính xác tổng quát của mô hình trên phần lớn dữ liệu thực tế.
  * **Ca Xấu nhất (`Worst Case`)**: Mẫu có kết quả thấp nhất, giúp phân tích các ca nhiễu phức tạp, biên mờ hoặc ranh giới khối u khó nhận diện để rút kinh nghiệm cải tiến.
* **Bảng trực quan hóa 5 cột đồng bộ (`5-Column Comparative Layout`)**:
  * **Cột 1 (`Ảnh MRI`)**: Lát cắt xám đầu vào sau khi tiền xử lý.
  * **Cột 2 (`Ground Truth`)**: Nhãn chuẩn vàng từ chuyên gia.
  * **Cột 3 (`Otsu + Post`)**: Kết quả từ phương pháp cơ sở truyền thống.
  * **Cột 4 (`U-Net + Post`)**: Kết quả từ mô hình học sâu đề xuất.
  * **Cột 5 (`Overlay`)**: Biểu đồ chồng lớp viền xanh (Ground Truth) và viền đỏ (U-Net Prediction), giúp đối chiếu trực diện sự chênh lệch ranh giới giữa hai phương pháp.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def truc_quan_hoa_best_median_worst(model, file_test, device):
    """
    Hàm tìm 3 ca (Tốt nhất, Trung vị, Xấu nhất) dựa trên Dice Score của U-Net + Hậu xử lý,
    sau đó vẽ biểu đồ 5 cột: Ảnh gốc | Ground Truth | Otsu | U-Net | Overlay
    """
    model.eval()
    danh_sach_ket_qua = []

    print("Đang quét toàn bộ tập Test để tìm ca Tốt nhất, Trung vị và Xấu nhất...")

    with torch.no_grad():
        for idx, duong_dan in enumerate(file_test):
            # 1. Đọc và tiền xử lý
            anh_goc, mask_goc = doc_file_mat(duong_dan)
            anh_xu_ly, mask_xu_ly = tien_xu_ly(anh_goc, mask_goc)

            # Chuẩn bị ảnh uint8 cho Otsu
            anh_uint8 = (anh_xu_ly * 255).astype(np.uint8) if anh_xu_ly.max() <= 1.0 else anh_xu_ly.astype(np.uint8)
            
            # Chạy phương pháp Otsu + Hậu xử lý
            pred_otsu = chay_otsu_va_hau_xu_ly(anh_uint8)

            # Chạy U-Net + Hậu xử lý
            anh_tensor = anh_xu_ly.astype(np.float32)
            if anh_tensor.max() > 1.0:
                anh_tensor = anh_tensor / 255.0
            anh_tensor = anh_tensor.reshape(1, 1, 256, 256)
            anh_tensor = torch.from_numpy(anh_tensor).to(device)

            dau_ra = model(anh_tensor)
            pred_unet_raw = (torch.sigmoid(dau_ra) > 0.5).float().squeeze().cpu().numpy()
            pred_unet = chay_unet_hau_xu_ly(pred_unet_raw)

            # Tính Dice Score của U-Net làm chuẩn xếp hạng
            dice_unet, _, _, _ = tinh_cac_chi_so(pred_unet, mask_xu_ly)

            # Lưu lại thông tin
            danh_sach_ket_qua.append({
                "index": idx,
                "path": duong_dan,
                "dice": dice_unet,
                "anh": anh_xu_ly,
                "mask_gt": mask_xu_ly,
                "pred_otsu": pred_otsu,
                "pred_unet": pred_unet
            })

    # 2. Sắp xếp danh sách theo Dice Score tăng dần
    danh_sach_ket_qua = sorted(danh_sach_ket_qua, key=lambda x: x["dice"])

    worst_case = danh_sach_ket_qua[0]                          # Xấu nhất (Min Dice)
    median_case = danh_sach_ket_qua[len(danh_sach_ket_qua) // 2] # Trung vị (Median Dice)
    best_case = danh_sach_ket_qua[-1]                         # Tốt nhất (Max Dice)

    print("\n" + "="*60)
    print("KẾT QUẢ TÌM KIẾM CÁC CA TIÊU BIỂU TRÊN TẬP TEST")
    print("="*60)
    print(f"Ca Tốt nhất (Best Case)   - Index: {best_case['index']} | Dice: {best_case['dice']:.4f}")
    print(f"Ca Trung vị (Median Case) - Index: {median_case['index']} | Dice: {median_case['dice']:.4f}")
    print(f"Ca Xấu nhất (Worst Case)  - Index: {worst_case['index']} | Dice: {worst_case['dice']:.4f}")
    print("="*60)

    # 3. Vẽ biểu đồ 5 cột cho 3 ca tiêu biểu
    cac_ca = [
        ("Ca Tốt nhất (Best)", best_case),
        ("Ca Trung vị (Median)", median_case),
        ("Ca Xấu nhất (Worst)", worst_case)
    ]

    for ten_tieu_de, case in cac_ca:
        plt.figure(figsize=(18, 4))
        plt.suptitle(f"{ten_tieu_de} (Dice Score: {case['dice']:.4f})", fontsize=14, fontweight='bold', y=1.05)

        # Cột 1: Ảnh MRI đầu vào
        plt.subplot(1, 5, 1)
        plt.imshow(case["anh"], cmap='gray')
        plt.title('1. Ảnh MRI')
        plt.axis('off')

        # Cột 2: Ground Truth (Mask thực tế)
        plt.subplot(1, 5, 2)
        plt.imshow(case["mask_gt"], cmap='gray')
        plt.title('2. Ground Truth')
        plt.axis('off')

        # Cột 3: Otsu + Hậu xử lý (Baseline)
        plt.subplot(1, 5, 3)
        plt.imshow(case["pred_otsu"], cmap='gray')
        plt.title('3. Otsu + Post')
        plt.axis('off')

        # Cột 4: U-Net + Hậu xử lý (Đề xuất)
        plt.subplot(1, 5, 4)
        plt.imshow(case["pred_unet"], cmap='gray')
        plt.title('4. U-Net + Post')
        plt.axis('off')

        # Cột 5: Overlay (Xanh: Ground Truth | Đỏ: Dự đoán U-Net)
        plt.subplot(1, 5, 5)
        plt.imshow(case["anh"], cmap='gray')
        plt.contour(case["mask_gt"], colors='blue', linewidths=1.2, levels=[0.5])
        plt.contour(case["pred_unet"], colors='red', linewidths=1.2, levels=[0.5])
        plt.title('5. Overlay\n(Xanh: GT | Đỏ: Pred)')
        plt.axis('off')

        plt.tight_layout()
        plt.show()

# --- CÁCH GỌI HÀM ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_eval = UNet(in_channels=1, out_channels=1).to(device)
model_eval.load_state_dict(torch.load('best_dice_model.pth', map_location=device))

# Lấy lại danh sách file test
danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
nhom = gom_theo_benh_nhan(danh_sach_file)
_, _, benh_nhan_test = chia_benh_nhan(nhom)
file_test = lay_file_cua_benh_nhan(benh_nhan_test, nhom)

# Chạy trực quan hóa
truc_quan_hoa_best_median_worst(model_eval, file_test, device)

# 4.Thống kê dữ liệu 

## 4.1 Phân Chia Dữ Liệu Theo Cấp Độ Bệnh Nhân & Thống Kê Tổng Quan (`Patient-Level Data Splitting & Statistics`)
Đoạn mã này thiết lập cấu trúc phân chia tập dữ liệu huấn luyện, kiểm định và kiểm tra, đồng thời thống kê minh bạch các thông số thực nghiệm cho đồ án:
* **Phân chia theo cấp độ bệnh nhân (`Patient-Level Split`)**: Sử dụng phương pháp gom nhóm theo mã định danh bệnh nhân (`gom_theo_benh_nhan`) trước khi chia tỷ lệ ($70\%$ Train, $15\%$ Val, $15\%$ Test). Điều này đảm bảo các lát cắt MRI của cùng một bệnh nhân không bị phân rải vào cả tập huấn luyện và tập kiểm tra, triệt tiêu hoàn toàn hiện tượng rò rỉ dữ liệu (`data leakage`) và giúp đánh giá chính xác khả năng khái quát hóa của mô hình.
* **Ánh xạ tệp dữ liệu tương ứng**: Triển khai hàm `lay_file_cua_benh_nhan` để trích xuất toàn bộ đường dẫn tệp ảnh và mặt nạ dựa trên danh sách bệnh nhân đã được phân loại ở bước trước.
* **Báo cáo thống kê trực quan**: In ra màn hình console bảng tổng kết chi tiết số lượng bệnh nhân và số lượng lát cắt MRI (`slices`) tương ứng cho từng tập, phục vụ cho việc trình bày phần phương pháp thực nghiệm trong báo cáo đồ án.

In [ ]:
# Lấy toàn bộ danh sách file và gom nhóm theo bệnh nhân
danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
nhom_benh_nhan = gom_theo_benh_nhan(danh_sach_file)

# Chia bệnh nhân theo tỷ lệ chuẩn của đồ án (thường là 70% Train, 15% Val, 15% Test)
bn_train, bn_val, bn_test = chia_benh_nhan(nhom_benh_nhan)

# Lấy danh sách file tương ứng cho từng tập
file_train = lay_file_cua_benh_nhan(bn_train, nhom_benh_nhan)
file_val = lay_file_cua_benh_nhan(bn_val, nhom_benh_nhan)
file_test = lay_file_cua_benh_nhan(bn_test, nhom_benh_nhan)

# In kết quả thống kê
print("="*50)
print("THỐNG KÊ PHÂN CHIA DỮ LIỆU ĐỒ ÁN")
print("="*50)
print(f"• Tập Train : {len(bn_train)} bệnh nhân | {len(file_train)} ảnh")
print(f"• Tập Val   : {len(bn_val)} bệnh nhân | {len(file_val)} ảnh")
print(f"• Tập Test  : {len(bn_test)} bệnh nhân | {len(file_test)} ảnh")
print("-" * 50)
print(f"TỔNG CỘNG: {len(bn_train) + len(bn_val) + len(bn_test)} bệnh nhân | {len(file_train) + len(file_val) + len(file_test)} ảnh")
print("="*50)

## 4.2 Thống Kê Chi Tiết Phân Bố Loại Khối U Theo Từng Tập Dữ Liệu (`Tumor Type Distribution Statistics`)
Đoạn mã này cung cấp công cụ phân tích sâu để kiểm tra sự cân bằng của tập dữ liệu Figshare thông qua việc bóc tách nhãn từ các tệp định dạng `.mat` (`h5py`):
* **Hàm thống kê chuyên biệt (`thong_ke_loai_u`)**: Duyệt qua từng bệnh nhân và từng lát cắt MRI trong nhóm tương ứng, trích xuất mã nhãn khối u từ cấu trúc tệp dữ liệu gốc (`f['cjdata']['label']`), phân loại chính xác theo 3 dạng tổn thương:
  1. **Meningioma** (U màng não - Nhãn `1`)
  2. **Glioma** (U thần kinh đệm - Nhãn `2`)
  3. **Pituitary** (U tuyến yên - Nhãn `3`)
* **Báo cáo phân bố định lượng**: Tự động tổng hợp và in ra màn hình console số lượng chi tiết của từng loại khối u trên cả ba tập `Train`, `Val`, và `Test`, giúp kiểm chứng tính đồng đều và minh bạch của chiến lược phân chia dữ liệu trong nghiên cứu.

In [ ]:
import numpy as np
import h5py
import os

# Lấy danh sách file và chia nhóm bệnh nhân
danh_sach_file = lay_danh_sach_file(THU_MUC_DU_LIEU)
nhom_benh_nhan = gom_theo_benh_nhan(danh_sach_file)
bn_train, bn_val, bn_test = chia_benh_nhan(nhom_benh_nhan)

# Hàm phụ trợ để đếm phân bố loại u (1: Meningioma, 2: Glioma, 3: Pituitary)
def thong_ke_loai_u(danh_sach_bn, nhom_dict):
    dem = {1: 0, 2: 0, 3: 0}
    tong_anh = 0
    for bn in danh_sach_bn:
        files = nhom_dict[bn]
        tong_anh += len(files)
        for f_path in files:
            with h5py.File(f_path, 'r') as f:
                label = int(f['cjdata']['label'][()][0][0])
                if label in dem:
                    dem[label] += 1
    return tong_anh, dem

tong_train, phan_bo_train = thong_ke_loai_u(bn_train, nhom_benh_nhan)
tong_val, phan_bo_val = thong_ke_loai_u(bn_val, nhom_benh_nhan)
tong_test, phan_bo_test = thong_ke_loai_u(bn_test, nhom_benh_nhan)

TEN_U = {1: "Meningioma", 2: "Glioma", 3: "Pituitary"}

print("="*65)
print("📊 THỐNG KÊ CHI TIẾT TẬP DỮ LIỆU ĐỒ ÁN")
print("="*65)
print(f"• Tập Train : {len(bn_train)} bệnh nhân | {tong_train} ảnh")
for k, v in phan_bo_train.items():
    print(f"    - {TEN_U[k]}: {v} ảnh")

print(f"• Tập Val   : {len(bn_val)} bệnh nhân | {tong_val} ảnh")
for k, v in phan_bo_val.items():
    print(f"    - {TEN_U[k]}: {v} ảnh")

print(f"• Tập Test  : {len(bn_test)} bệnh nhân | {tong_test} ảnh")
for k, v in phan_bo_test.items():
    print(f"    - {TEN_U[k]}: {v} ảnh")
print("="*65)

## 4.3 Xây Dựng Pipeline Trực Quan Hóa Quy Trình Tiền Xử Lý & Tăng Cường Dữ Liệu (`Data Preprocessing & Augmentation Visualization`)
Đoạn mã này cung cấp công cụ trực quan hóa toàn bộ chuỗi các bước biến đổi dữ liệu đầu vào trước khi đưa vào mô hình học sâu, giúp kiểm chứng chất lượng chuẩn hóa và tính hợp lý của kỹ thuật tăng cường dữ liệu (`Data Augmentation`):
* **Đọc và nạp dữ liệu gốc**: Truy xuất tệp `.mat` thông qua đường dẫn chỉ định, trích xuất đồng thời lát cắt MRI gốc và mặt nạ vùng tổn thương (`Ground Truth`).
* **Trực quan hóa 4 bước tuần tự (`4-Step Data Pipeline Layout`)**:
  1. **Ảnh Gốc & Viền U (`Raw Image & Tumor Contour`)**: Hiển thị ảnh thô kèm theo đường viền đỏ phác họa ranh giới khối u từ nhãn bác sĩ.
  2. **Resize & Chuẩn Hóa (`Resizing & Normalization`)**: Thể hiện kết quả sau khi chuẩn hóa kích thước không gian (`KICH_THUOC x KICH_THUOC`) và đồng nhất dải giá trị cường độ sáng.
  3. **Khử Nhiễu (`Noise Filtering`)**: Minh họa trạng thái ảnh sau khi áp dụng các bộ lọc làm mượt không gian nhằm triệt tiêu nhiễu nền.
  4. **Tăng Cường Dữ Liệu (`Data Augmentation`)**: Minh họa các phép biến đổi hình học ngẫu nhiên (như lật ngang `np.fliplr` hoặc xoay nhẹ) kết hợp cùng mặt nạ tương ứng (đường viền xanh dương), giúp mở rộng không gian mẫu và nâng cao khả năng chống quá khớp cho mô hình U-Net.

In [ ]:
import matplotlib.pyplot as plt

def truc_quan_hoa_tien_xu_ly(duong_dan_file):
    # 1. Đọc ảnh gốc và mask từ file .mat
    anh_goc, mask_goc = doc_file_mat(duong_dan_file)
    
    # 2. Tiền xử lý (Resize, khử nhiễu, chuẩn hóa)
    anh_xu_ly, mask_xu_ly = tien_xu_ly(anh_goc, mask_goc)
    
    # Vẽ biểu đồ các bước
    plt.figure(figsize=(15, 4))
    
    # Bước 1: Ảnh gốc
    plt.subplot(1, 4, 1)
    plt.imshow(anh_goc, cmap='gray')
    if np.any(mask_goc > 0):
        plt.contour(mask_goc, colors='red', linewidths=1)
    plt.title('1. Ảnh Gốc & Viền U')
    plt.axis('off')
    
    # Bước 2: Sau tiền xử lý & Chuẩn hóa
    plt.subplot(1, 4, 2)
    plt.imshow(anh_xu_ly, cmap='gray')
    if np.any(mask_xu_ly > 0):
        plt.contour(mask_xu_ly, colors='red', linewidths=1)
    plt.title(f'2. Resize & Chuẩn hóa\n({KICH_THUOC}x{KICH_THUOC})')
    plt.axis('off')
    
    # Bước 3: Minh họa Lọc nhiễu / Hậu xử lý
    plt.subplot(1, 4, 3)
    plt.imshow(anh_xu_ly, cmap='gray')
    plt.title('3. Khử nhiễu (Filtering)')
    plt.axis('off')
    
    # Bước 4: Minh họa Data Augmentation (Ví dụ lật ngang/xoay nhẹ)
    anh_aug = np.fliplr(anh_xu_ly)
    mask_aug = np.fliplr(mask_xu_ly)
    plt.subplot(1, 4, 4)
    plt.imshow(anh_aug, cmap='gray')
    if np.any(mask_aug > 0):
        plt.contour(mask_aug, colors='blue', linewidths=1)
    plt.title('4. Augmentation\n(Lật ngang / Xoay)')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Gọi thử với file đầu tiên trong tập test
truc_quan_hoa_tien_xu_ly(file_test[0])

---

# 5.Thảo Luận: Hạn Chế Của Nghiên Cứu (`Limitations`)
Mặc dù mô hình U-Net kết hợp hậu xử lý hình thái học đã đạt được hiệu năng phân đoạn vượt trội so với phương pháp Otsu truyền thống, nghiên cứu vẫn tồn tại một số điểm hạn chế cần nhìn nhận khách quan:
* **Độ nhạy với biên mờ và khối u kích thước nhỏ**: Ở các ca thuộc nhóm "Xấu nhất" (`Worst Case`), nơi ranh giới khối u không rõ ràng hoặc có sự hòa lẫn cường độ sáng với các mô nền xung quanh, mô hình đôi khi gặp khó khăn trong việc xác định chính xác đường biên, dẫn đến sụt giảm nhẹ hệ số Dice Score.
* **Giới hạn của mô hình 2D**: Việc xử lý từng lát cắt 2D độc lập giúp tối ưu hóa tài nguyên tính toán nhưng chưa khai thác triệt để mối quan hệ không gian ba chiều (`3D spatial context`) liên tục giữa các lát cắt MRI kế cận.
* **Sự phụ thuộc vào chất lượng tiền xử lý**: Hiệu năng của cả phương pháp Otsu lẫn U-Net đều chịu ảnh hưởng bởi nhiễu thiết bị và độ tương phản của ảnh gốc, đòi hỏi các bước chuẩn hóa cường độ sáng rất khắt khe.

---

# 6 Hướng Phát Triển Tương Lai (`Future Work`)
Dựa trên những hạn chế trên, một số hướng cải tiến và mở rộng tiềm năng cho các nghiên cứu tiếp theo bao gồm:
* **Nâng cấp lên kiến trúc 3D U-Net / V-Net**: Chuyển đổi từ mô hình xử lý từng lát cắt 2D sang không gian 3D nguyên bản nhằm tận dụng toàn diện thông tin cấu trúc thể tích khối u qua các lớp MRI liên tiếp.
* **Ứng dụng các hàm mất mát nâng cao (`Advanced Loss Functions`)**: Thử nghiệm kết hợp thêm *Focal Tversky Loss* hoặc *Boundary Loss* để giải quyết triệt để bài toán mất cân bằng điểm ảnh giữa vùng u nhỏ và nền mô lành.
* **Tích hợp cơ chế Attention Gate**: Bổ sung các khối chú ý không gian (`Attention U-Net`) giúp mô hình tự động tập trung trọng số vào vùng tổn thương và triệt tiêu các vùng nhiễu nền không liên quan.

---

# 7 Kết Luận Đồ Án (`Conclusion`)
Đồ án đã hoàn thành xuất sắc việc xây dựng và hiện thực hóa toàn bộ quy trình phân đoạn khối u toàn diện trên tập dữ liệu *Figshare* trên nền tảng Kaggle:
* **Về mặt phương pháp luận**: Đã triển khai thành công phương pháp cơ sở Otsu truyền thống kết hợp hậu xử lý hình thái học và lọc thành phần liên thông, đồng thời đề xuất mô hình học sâu **U-Net** tối ưu với hàm mất mát hỗn hợp `DiceBCELoss`.
* **Về mặt định lượng**: Qua quá trình đánh giá độc lập trên tập kiểm tra (`Test Set`), mô hình U-Net đề xuất đã chứng minh sự vượt trội tuyệt đối, nâng cao đáng kể các chỉ số đánh giá y khoa (đặc biệt là hệ số tương đồng *Dice Coefficient* và độ đo *IoU*) so với các phương pháp phân ngưỡng truyền thống.
* **Về mặt ứng dụng thực tiễn**: Hệ thống cung cấp bộ mã nguồn dạng mô-đun hóa cao (`modular code`), tích hợp trực quan hóa 4 cột và 5 cột (so sánh Best, Median, Worst Case), tạo nền tảng vững chắc cho việc hỗ trợ chẩn đoán hình ảnh y khoa tự động trong tương lai.